 INSTRUCTIONS

In [ ]:
!pip install praw


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 3.9 MB/s eta 0:00:00


# **FINAL DRAFT** Overrall

In [ ]:
import praw
import pandas as pd
import time
import gspread
from google.oauth2.service_account import Credentials
import spacy
import datetime
import re
from collections import Counter
import requests
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download required NLTK data
try:
    nltk.download('vader_lexicon', quiet=True)
except:
    pass

# --- Configuration ---
SERVICE_ACCOUNT_FILE = 'YOUR_SERVICE_ACCOUNT_FILE'
MOVIE_SHEET_ID = 'YOUR_MOVIE_SHEET_ID'
MOVIE_WORKSHEET_NAME = 'movies'
NAME_COLUMN_IN_GSHEET = 'Movie Names'

SUBREDDIT_NAME = 'movies'
POST_LIMIT = 100  # Number of posts WITH detected movies to collect
COMMENTS_PER_POST = 3
OUTPUT_CSV = 'r_movies_collected_data.csv'
MAX_POSTS_TO_SCAN = 500  # Maximum posts to scan to find POST_LIMIT with movies

# Reddit API Configuration
REDDIT_CONFIG = {
    'client_id': 'YOUR_CLIENT_ID',
    'client_secret': "YOUR_CLIENT_SECRET",
    'user_agent': 'MovieSentimentBot by /u/anonymous'
}

# Expanded movie database with popular movies, franchises, and series
COMPREHENSIVE_MOVIES = [
    # Marvel Cinematic Universe
    "Iron Man", "Captain America", "Thor", "The Avengers", "Hulk", "Black Widow",
    "Guardians of the Galaxy", "Ant-Man", "Doctor Strange", "Spider-Man", "Black Panther",
    "Captain Marvel", "Avengers Endgame", "Infinity War", "Civil War", "Winter Soldier",
    "Age of Ultron", "Ragnarok", "Homecoming", "Far From Home", "No Way Home",
    "Eternals", "Shang-Chi", "Multiverse of Madness", "Love and Thunder", "Wakanda Forever",

    # DC Universe
    "Batman", "Superman", "Wonder Woman", "Justice League", "Aquaman", "The Flash",
    "Green Lantern", "Suicide Squad", "Birds of Prey", "Joker", "The Batman",
    "Man of Steel", "Batman v Superman", "Dark Knight", "Dark Knight Rises", "Batman Begins",

    # Star Wars
    "Star Wars", "Empire Strikes Back", "Return of the Jedi", "Phantom Menace",
    "Attack of the Clones", "Revenge of the Sith", "Force Awakens", "Last Jedi",
    "Rise of Skywalker", "Rogue One", "Solo", "Mandalorian",

    # Classic/Popular Films
    "The Shawshank Redemption", "The Godfather", "Pulp Fiction", "Forrest Gump",
    "Inception", "The Matrix", "Goodfellas", "Casablanca", "Citizen Kane",
    "Schindler's List", "Lord of the Rings", "Fellowship of the Ring", "Two Towers",
    "Return of the King", "The Hobbit", "Titanic", "Avatar", "Jurassic Park",
    "E.T.", "Jaws", "Back to the Future", "Indiana Jones", "Raiders of the Lost Ark",

    # Horror
    "Halloween", "Friday the 13th", "Nightmare on Elm Street", "The Exorcist",
    "The Shining", "Psycho", "Scream", "Get Out", "Hereditary", "Midsommar",
    "It", "The Conjuring", "Insidious", "Paranormal Activity", "Saw", "Texas Chainsaw Massacre",

    # Action/Thriller
    "John Wick", "Fast and Furious", "Mission Impossible", "Die Hard", "Lethal Weapon",
    "Terminator", "Alien", "Predator", "Rocky", "Rambo", "Mad Max", "Blade Runner",
    "The Bourne Identity", "James Bond", "Casino Royale", "Skyfall", "Spectre",

    # Comedy
    "Anchorman", "Superbad", "Pineapple Express", "Step Brothers", "Talladega Nights",
    "Zoolander", "Meet the Parents", "There's Something About Mary", "Dumb and Dumber",
    "The Hangover", "Bridesmaids", "Ghostbusters", "Groundhog Day", "Big Lebowski",

    # Drama
    "The Departed", "Scarface", "Casino", "Taxi Driver", "Raging Bull",
    "The Wolf of Wall Street", "Django Unchained", "Kill Bill", "Inglourious Basterds",
    "Once Upon a Time in Hollywood", "Parasite", "Nomadland", "Minari", "Sound of Metal",

    # Animated
    "Toy Story", "Finding Nemo", "The Incredibles", "Up", "Wall-E", "Inside Out",
    "Coco", "Moana", "Frozen", "The Lion King", "Beauty and the Beast", "Aladdin",
    "Shrek", "How to Train Your Dragon", "Despicable Me", "Minions", "Ice Age",

    # Recent Popular
    "Top Gun Maverick", "Everything Everywhere All at Once", "The Batman", "Doctor Strange",
    "Thor Love and Thunder", "Black Panther Wakanda Forever", "Avatar Way of Water",
    "Glass Onion", "Knives Out", "Don't Look Up", "Dune", "No Time to Die", "Spider-Man No Way Home"
]

# Movie abbreviations and slang
MOVIE_ABBREVIATIONS = {
    'mcu': 'marvel cinematic universe',
    'dceu': 'dc extended universe',
    'lotr': 'lord of the rings',
    'sw': 'star wars',
    'hp': 'harry potter',
    'gotg': 'guardians of the galaxy',
    'potc': 'pirates of the caribbean',
    'tdk': 'the dark knight',
    'tdkr': 'the dark knight rises',
    'bb': 'batman begins',
    'bvs': 'batman v superman',
    'mos': 'man of steel',
    'jl': 'justice league',
    'ss': 'suicide squad',
    'ww': 'wonder woman',
    'am': 'aquaman',
    'sm': 'spider-man',
    'im': 'iron man',
    'ca': 'captain america',
    'tws': 'the winter soldier',
    'cw': 'civil war',
    'aou': 'age of ultron',
    'tfa': 'the force awakens',
    'tlj': 'the last jedi',
    'tros': 'the rise of skywalker',
    'rotj': 'return of the jedi',
    'esb': 'empire strikes back',
    'anh': 'a new hope',
    'rots': 'revenge of the sith',
    'aotc': 'attack of the clones',
    'tpm': 'the phantom menace',
    'iw': 'infinity war',
    'eg': 'endgame',
    'nwh': 'no way home',
    'ffh': 'far from home',
    'hc': 'homecoming',
    'ragnarok': 'thor ragnarok',
    'bp': 'black panther',
    'cm': 'captain marvel',
    'ds': 'doctor strange',
    'gotg2': 'guardians of the galaxy vol 2',
    'antman': 'ant-man',
    'jp': 'jurassic park',
    'bttf': 'back to the future',
    'rotla': 'raiders of the lost ark',
    'mi': 'mission impossible'
}

# Minimal blacklist - only truly generic words
STRICT_BLACKLIST = {
    'movie', 'movies', 'film', 'films', 'cinema', 'flick', 'picture', 'show',
    'the', 'a', 'an', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with',
    'this', 'that', 'my', 'your', 'his', 'her', 'our', 'their', 'it', 'its'
}

# Movie context indicators - words that suggest movie discussion
MOVIE_CONTEXT_WORDS = {
    'watch', 'watched', 'watching', 'see', 'seen', 'saw', 'love', 'loved', 'hate', 'hated',
    'like', 'liked', 'dislike', 'enjoy', 'enjoyed', 'favorite', 'favourite', 'fav',
    'best', 'worst', 'good', 'bad', 'great', 'amazing', 'terrible', 'awful',
    'recommend', 'recommended', 'suggest', 'review', 'reviewed', 'rating', 'rated',
    'director', 'directed', 'starring', 'stars', 'cast', 'actor', 'actress',
    'sequel', 'prequel', 'remake', 'reboot', 'trailer', 'teaser', 'clip',
    'spoiler', 'spoilers', 'plot', 'story', 'character', 'characters',
    'scene', 'scenes', 'ending', 'beginning', 'climax', 'twist'
}

class EmotionAnalyzer:
    def __init__(self):
        """Initialize emotion analysis tools"""
        try:
            self.sia = SentimentIntensityAnalyzer()
        except:
            print("Warning: VADER sentiment analyzer not available")
            self.sia = None

    def analyze_emotions(self, text):
        """Analyze emotions in text and return top 3 emotions with scores"""
        if not text or not isinstance(text, str):
            return {
                'emotion_1': 'neutral',
                'emotion_1_score': 0.0,
                'emotion_2': 'neutral',
                'emotion_2_score': 0.0,
                'emotion_3': 'neutral',
                'emotion_3_score': 0.0
            }

        # Clean text for analysis
        text = text.lower().strip()

        # Get basic sentiment using TextBlob
        try:
            blob = TextBlob(text)
            polarity = blob.sentiment.polarity  # -1 to 1
            subjectivity = blob.sentiment.subjectivity  # 0 to 1
        except:
            polarity = 0.0
            subjectivity = 0.0

        # Get VADER sentiment scores
        if self.sia:
            try:
                vader_scores = self.sia.polarity_scores(text)
                compound = vader_scores['compound']
                pos_score = vader_scores['pos']
                neg_score = vader_scores['neg']
                neu_score = vader_scores['neu']
            except:
                compound = pos_score = neg_score = neu_score = 0.0
        else:
            compound = pos_score = neg_score = neu_score = 0.0

        # Emotion keyword detection with scores
        emotions = self._detect_emotion_keywords(text)

        # Combine different approaches to determine top emotions
        emotion_scores = {
            'joy': max(pos_score, polarity if polarity > 0 else 0) + emotions.get('joy', 0),
            'excitement': emotions.get('excitement', 0) + (pos_score * 0.8),
            'love': emotions.get('love', 0) + (pos_score * 0.6),
            'anger': neg_score + emotions.get('anger', 0),
            'disappointment': emotions.get('disappointment', 0) + (neg_score * 0.7),
            'fear': emotions.get('fear', 0) + (neg_score * 0.5),
            'sadness': emotions.get('sadness', 0) + (neg_score * 0.6),
            'surprise': emotions.get('surprise', 0) + (abs(polarity) * subjectivity),
            'nostalgia': emotions.get('nostalgia', 0),
            'confusion': emotions.get('confusion', 0) + (subjectivity * 0.5),
            'anticipation': emotions.get('anticipation', 0),
            'neutral': neu_score
        }

        # Get top 3 emotions
        sorted_emotions = sorted(emotion_scores.items(), key=lambda x: x[1], reverse=True)
        top_3 = sorted_emotions[:3]

        # Ensure we have 3 emotions
        while len(top_3) < 3:
            top_3.append(('neutral', 0.0))

        return {
            'emotion_1': top_3[0][0],
            'emotion_1_score': round(top_3[0][1], 3),
            'emotion_2': top_3[1][0],
            'emotion_2_score': round(top_3[1][1], 3),
            'emotion_3': top_3[2][0],
            'emotion_3_score': round(top_3[2][1], 3)
        }

    def _detect_emotion_keywords(self, text):
        """Detect emotion-specific keywords and return scores"""
        emotion_keywords = {
            'joy': ['amazing', 'awesome', 'fantastic', 'incredible', 'brilliant', 'outstanding',
                   'excellent', 'wonderful', 'perfect', 'beautiful', 'hilarious', 'funny'],
            'excitement': ['excited', 'thrilled', 'pumped', 'hyped', 'anticipating', 'cant wait',
                          'looking forward', 'epic', 'incredible', 'mind-blowing'],
            'love': ['love', 'adore', 'favorite', 'favourite', 'obsessed', 'amazing chemistry',
                    'perfect cast', 'masterpiece', 'classic'],
            'anger': ['hate', 'terrible', 'awful', 'worst', 'horrible', 'disgusting', 'garbage',
                     'trash', 'ruined', 'destroyed', 'furious', 'angry'],
            'disappointment': ['disappointed', 'letdown', 'expected better', 'underwhelming',
                              'waste of time', 'boring', 'meh', 'overrated', 'overhyped'],
            'fear': ['scared', 'terrifying', 'creepy', 'disturbing', 'nightmare', 'horrifying',
                    'spine-chilling', 'jump scare', 'suspenseful'],
            'sadness': ['sad', 'depressing', 'heartbreaking', 'emotional', 'tear jerker', 'cried',
                       'touching', 'moving', 'tragic'],
            'surprise': ['shocked', 'unexpected', 'plot twist', 'surprised', 'didnt see coming',
                        'jaw dropped', 'mind blown', 'revelation'],
            'nostalgia': ['nostalgic', 'childhood', 'memories', 'throwback', 'classic', 'old school',
                         'brings back', 'remember when'],
            'confusion': ['confused', 'wtf', 'what the', 'makes no sense', 'plot holes',
                         'convoluted', 'complicated', 'lost me'],
            'anticipation': ['cant wait', 'excited for', 'looking forward', 'hoping', 'sequel',
                           'part 2', 'next movie', 'continuation']
        }

        emotion_scores = {}
        words = text.split()

        for emotion, keywords in emotion_keywords.items():
            score = 0
            for keyword in keywords:
                # Count occurrences of keywords
                if keyword in text:
                    score += text.count(keyword) * 0.2

                # Check for partial matches in words
                for word in words:
                    if keyword in word:
                        score += 0.1

            emotion_scores[emotion] = min(score, 1.0)  # Cap at 1.0

        return emotion_scores

class AdvancedMovieDetector:
    def __init__(self):
        print("Initializing enhanced movie detection system...")
        # Load spaCy with only essential components
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
        self.movie_titles = set()
        self.movie_patterns = []
        self.setup_detection_system()

    def setup_detection_system(self):
        """Setup comprehensive movie detection"""
        # Load all movies
        all_movies = COMPREHENSIVE_MOVIES.copy()

        # Try to load from Google Sheets if available
        try:
            additional_movies = self._load_from_google_sheets()
            if additional_movies:
                all_movies.extend(additional_movies)
        except:
            pass

        # Process all movie titles
        self._process_all_movies(all_movies)

        # Setup EntityRuler with aggressive patterns
        self._setup_entity_ruler()

        print(f"Detection system ready with {len(self.movie_titles)} movie patterns")

    def _load_from_google_sheets(self):
        """Try to load additional movies from Google Sheets"""
        try:
            import os
            if not os.path.exists(SERVICE_ACCOUNT_FILE):
                return None

            scopes = ['https://www.googleapis.com/auth/spreadsheets']
            creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scopes)
            client = gspread.authorize(creds)
            sheet = client.open_by_key(MOVIE_SHEET_ID).worksheet(MOVIE_WORKSHEET_NAME)
            all_data = sheet.get_all_records()
            df = pd.DataFrame(all_data)

            if NAME_COLUMN_IN_GSHEET in df.columns:
                movies = df[NAME_COLUMN_IN_GSHEET].dropna().astype(str).tolist()
                print(f"Loaded {len(movies)} additional movies from Google Sheets")
                return movies
        except Exception as e:
            print(f"Could not load from Google Sheets: {e}")
        return None

    def _process_all_movies(self, movies):
        """Process and clean all movie titles"""
        for movie in movies:
            if not movie or len(str(movie).strip()) < 2:
                continue

            movie = str(movie).strip()

            # Add original title
            cleaned = self._clean_title(movie)
            if cleaned and self._is_valid_movie_title(cleaned):
                self.movie_titles.add(cleaned.lower())

                # Add variations
                self._add_variations(cleaned)

        # Add abbreviations
        for abbr, full_name in MOVIE_ABBREVIATIONS.items():
            self.movie_titles.add(abbr.lower())
            self.movie_titles.add(full_name.lower())

    def _clean_title(self, title):
        """Clean movie title"""
        # Remove year in parentheses
        title = re.sub(r'\s*\(\d{4}\).*$', '', title)

        # Remove common suffixes
        title = re.sub(r'\s+(movie|film)$', '', title, flags=re.IGNORECASE)

        # Clean special characters but keep apostrophes and hyphens
        title = re.sub(r'[^\w\s\'\-]', ' ', title)

        # Normalize whitespace
        title = ' '.join(title.split())

        return title.strip()

    def _is_valid_movie_title(self, title):
        """Check if title is valid - very permissive now"""
        title_lower = title.lower().strip()

        # Only reject if it's in the strict blacklist
        if title_lower in STRICT_BLACKLIST:
            return False

        # Must be at least 2 characters
        if len(title_lower) < 2:
            return False

        return True

    def _add_variations(self, title):
        """Add common variations"""
        title_lower = title.lower()

        # Add version without "the"
        if title_lower.startswith('the '):
            self.movie_titles.add(title_lower[4:])

        # Add version without apostrophes
        if "'" in title_lower:
            self.movie_titles.add(title_lower.replace("'", ""))
            self.movie_titles.add(title_lower.replace("'", ""))

        # Add hyphenated version
        if ' ' in title_lower and len(title_lower.split()) == 2:
            self.movie_titles.add(title_lower.replace(' ', '-'))

    def _setup_entity_ruler(self):
        """Setup EntityRuler with comprehensive patterns"""
        ruler = self.nlp.add_pipe("entity_ruler", before="ner")

        patterns = []
        for movie in self.movie_titles:
            patterns.append({"label": "MOVIE", "pattern": movie})

        ruler.add_patterns(patterns)
        print(f"Added {len(patterns)} patterns to EntityRuler")

    def detect_movies_comprehensive(self, text):
        """Comprehensive movie detection using multiple strategies"""
        if not text or not isinstance(text, str):
            return []

        text = text.lower().strip()
        found_movies = set()

        # Strategy 1: spaCy EntityRuler
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ == "MOVIE":
                movie = ent.text.strip()
                if movie in MOVIE_ABBREVIATIONS:
                    movie = MOVIE_ABBREVIATIONS[movie]
                found_movies.add(movie.title())

        # Strategy 2: Direct string matching with context
        found_movies.update(self._contextual_string_matching(text))

        # Strategy 3: Quoted text detection
        found_movies.update(self._detect_quoted_titles(text))

        # Strategy 4: Capitalized sequences
        found_movies.update(self._detect_capitalized_sequences(text))

        # Filter and validate results
        validated_movies = []
        for movie in found_movies:
            if self._validate_detection(movie, text):
                validated_movies.append(movie)

        return list(set(validated_movies))  # Remove duplicates

    def _contextual_string_matching(self, text):
        """Find movies using context-aware string matching"""
        found = set()
        words = text.split()

        for i, word in enumerate(words):
            # Check if current word or phrase matches a movie
            for length in range(1, min(6, len(words) - i + 1)):  # Check up to 5-word phrases
                phrase = ' '.join(words[i:i+length])

                if phrase in self.movie_titles:
                    # Check for movie context nearby
                    context_start = max(0, i - 5)
                    context_end = min(len(words), i + length + 5)
                    context = ' '.join(words[context_start:context_end])

                    if any(ctx_word in context for ctx_word in MOVIE_CONTEXT_WORDS):
                        # Expand abbreviations
                        if phrase in MOVIE_ABBREVIATIONS:
                            phrase = MOVIE_ABBREVIATIONS[phrase]
                        found.add(phrase.title())

        return found

    def _detect_quoted_titles(self, text):
        """Detect movie titles in quotes"""
        found = set()

        # Find text in quotes
        quote_patterns = [
            r'"([^"]{2,50})"',  # Double quotes
            r"'([^']{2,50})'",  # Single quotes
            r'\"([^\"]{2,50})\"'  # Escaped quotes
        ]

        for pattern in quote_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if match.lower().strip() in self.movie_titles:
                    found.add(match.strip().title())

        return found

    def _detect_capitalized_sequences(self, text):
        """Detect capitalized word sequences that might be movie titles"""
        found = set()

        # Find sequences of capitalized words
        cap_pattern = r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b'
        matches = re.findall(cap_pattern, text)

        for match in matches:
            if match.lower() in self.movie_titles:
                found.add(match.title())

        return found

    def _validate_detection(self, movie, full_text):
        """Validate movie detection - very permissive"""
        if not movie or len(movie.strip()) < 2:
            return False

        movie_lower = movie.lower().strip()

        # Reject only strict blacklist items
        if movie_lower in STRICT_BLACKLIST:
            return False

        # Accept if it's a known abbreviation
        if movie_lower in MOVIE_ABBREVIATIONS:
            return True

        # Accept if it's in our movie database
        if movie_lower in self.movie_titles:
            return True

        # Accept multi-word titles (more likely to be specific)
        if len(movie.split()) > 1:
            return True

        return False

class RedditScraper:
    def __init__(self, reddit_config):
        self.reddit = praw.Reddit(**reddit_config)
        self.emotion_analyzer = EmotionAnalyzer()

    def collect_data(self, subreddit_name, post_limit, comments_per_post, movie_detector, max_posts_to_scan):
        """Collect Reddit data with enhanced movie detection and emotion analysis - skip posts without movies"""
        all_data = []
        posts_with_movies = 0
        posts_scanned = 0
        skipped_posts = 0

        print(f"\nScraping {post_limit} posts WITH movies from r/{subreddit_name}...")
        print(f"Will scan up to {max_posts_to_scan} posts to find {post_limit} with detected movies")

        try:
            subreddit = self.reddit.subreddit(subreddit_name)

            for submission in subreddit.hot(limit=max_posts_to_scan):
                posts_scanned += 1

                # Check if we've found enough posts with movies
                if posts_with_movies >= post_limit:
                    break

                # Combine title and content for analysis
                post_text = f"{submission.title}. {submission.selftext}".strip()

                # Detect movies in post
                found_movies = movie_detector.detect_movies_comprehensive(post_text)

                # Also check comments for additional movies
                comment_text = ""
                comment_movies = []

                if submission.num_comments > 0:
                    try:
                        submission.comments.replace_more(limit=0)
                        comments = [c.body for c in submission.comments[:comments_per_post]
                                  if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']]
                        comment_text = " ".join(comments[:3])  # Limit comment text

                        if comment_text:
                            comment_movies = movie_detector.detect_movies_comprehensive(comment_text)

                    except Exception as e:
                        print(f"Error processing comments for post {posts_scanned}: {e}")

                # Combine all found movies
                all_found_movies = list(set(found_movies + comment_movies))

                # Skip this post if no movies were detected
                if not all_found_movies:
                    skipped_posts += 1
                    if posts_scanned % 20 == 0:
                        print(f"Scanned {posts_scanned} posts | Found {posts_with_movies}/{post_limit} with movies | Skipped {skipped_posts}")
                    continue

                # We found movies! Process this post
                posts_with_movies += 1

                # Analyze emotions in the combined text
                full_text_for_emotion = f"{post_text} {comment_text}"
                emotion_results = self.emotion_analyzer.analyze_emotions(full_text_for_emotion)

                # Store results (no 'Unknown' movies anymore)
                data_entry = {
                    'Title': submission.title,
                    'Author': str(submission.author) if submission.author else '[deleted]',
                    'All_Movies': all_found_movies,
                    'Primary_Movie': all_found_movies[0],  # Always has at least one movie
                    'Movie_Count': len(all_found_movies),
                    'Post_Movies': found_movies,
                    'Comment_Movies': comment_movies,
                    'Content': submission.selftext[:200] + '...' if len(submission.selftext) > 200 else submission.selftext,
                    'Comments_Sample': comment_text[:300] + '...' if len(comment_text) > 300 else comment_text,
                    'Score': submission.score,
                    'Num_Comments': submission.num_comments,
                    'Timestamp_UTC': datetime.datetime.utcfromtimestamp(submission.created_utc),
                    'URL': f"https://reddit.com{submission.permalink}"
                }

                # Add emotion analysis results
                data_entry.update(emotion_results)
                all_data.append(data_entry)

                if posts_with_movies % 10 == 0:
                    print(f"Progress: {posts_with_movies}/{post_limit} posts with movies | Scanned {posts_scanned} total | Skipped {skipped_posts}")

                time.sleep(0.2)  # Rate limiting

        except Exception as e:
            print(f"Error during scraping: {e}")

        print(f"\nScraping complete!")
        print(f"Posts scanned: {posts_scanned}")
        print(f"Posts with movies found: {posts_with_movies}")
        print(f"Posts skipped (no movies): {skipped_posts}")
        print(f"Success rate: {posts_with_movies/posts_scanned*100:.1f}%")

        return pd.DataFrame(all_data)

def main():
    """Main execution function"""
    try:
        # Initialize enhanced movie detector
        detector = AdvancedMovieDetector()

        # Initialize Reddit scraper
        scraper = RedditScraper(REDDIT_CONFIG)

        # Collect data - only posts with detected movies
        df = scraper.collect_data(SUBREDDIT_NAME, POST_LIMIT, COMMENTS_PER_POST, detector, MAX_POSTS_TO_SCAN)

        # Save results
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\nData saved to {OUTPUT_CSV}")

        # Show comprehensive results
        print(f"\n=== DETECTION RESULTS ===")
        print(f"Total posts with movies collected: {len(df)}")

        # Movie statistics - all posts now have movies
        all_movies = []
        for movies_list in df['All_Movies']:
            if isinstance(movies_list, list):
                all_movies.extend(movies_list)

        if all_movies:
            movie_counts = Counter(all_movies)
            print(f"\nTop 15 detected movies:")
            for movie, count in movie_counts.most_common(15):
                print(f"  {movie}: {count}")

        # Show sample detections with emotions
        if not df.empty:
            print(f"\nSample movie detections with emotions:")
            sample = df[['Title', 'Primary_Movie', 'Movie_Count', 'emotion_1', 'emotion_1_score', 'Score']].head(10)
            for _, row in sample.iterrows():
                print(f"  '{row['Title'][:50]}...' -> {row['Primary_Movie']} | Top emotion: {row['emotion_1']} ({row['emotion_1_score']})")

        # Show emotion statistics
        if not df.empty:
            print(f"\nTop emotions detected:")
            all_emotions = list(df['emotion_1']) + list(df['emotion_2']) + list(df['emotion_3'])
            emotion_counts = Counter(all_emotions)
            for emotion, count in emotion_counts.most_common(10):
                print(f"  {emotion}: {count}")

        # Show posts with multiple movies
        multi_movie_df = df[df['Movie_Count'] > 1]
        if not multi_movie_df.empty:
            print(f"\nPosts with multiple movies detected: {len(multi_movie_df)}")
            print(f"Average movies per post: {df['Movie_Count'].mean():.1f}")

        print(f"\n=== ANALYSIS COMPLETE ===")
        print(f"Dataset contains {len(df)} posts, all with detected movies")
        print(f"Data includes movie detection + top 3 emotions with scores for each post")
        print(f"No 'Unknown' entries - only posts with successfully detected movies are included")

    except Exception as e:
        print(f"Script failed: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

Initializing enhanced movie detection system...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Added 312 patterns to EntityRuler
Detection system ready with 312 movie patterns

Scraping 100 posts WITH movies from r/movies...
Will scan up to 500 posts to find 100 with detected movies


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 10/100 posts with movies | Scanned 15 total | Skipped 5


...

Progress: 80/100 posts with movies | Scanned 129 total | Skipped 49


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 90/100 posts with movies | Scanned 140 total | Skipped 50


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 100/100 posts with movies | Scanned 159 total | Skipped 59

Scraping complete!
Posts scanned: 160
Posts with movies found: 100
Posts skipped (no movies): 59
Success rate: 62.5%

Data saved to r_movies_collected_data.csv

=== DETECTION RESULTS ===
Total posts with movies collected: 100

Top 15 detected movies:
  Up: 48
  Captain America: 28
  Aquaman: 18
  Saw: 17
  Lord Of The Rings: 5
  Iron Man: 4
  Spider-Man: 3
  The Lion King: 3
  The Shawshank Redemption: 2
  Big Lebowski: 2
  Rocky: 2
  Blade Runner: 2
  Solo: 2
  Star Wars: 2
  Titanic: 2

Sample movie detections with emotions:
  'AMA/Q&A Announcement - Stephen King - Wednesday 8/...' -> The Shawshank Redemption | Top emotion: neutral (0.852)
  'During the development of the Harriet Tubman biopi...' -> Spider-Man | Top emotion: joy (1.02)
  'Eddie Murphy Turned Down ‘Rush Hour’ To Go To Miam...' -> Lethal Weapon | Top emotion: neutral (0.885)
  'An American In Paris, 1951, Gene Kelly dancing is ...' -> Aquaman | Top e

# **Final Code r/movies**

### CODE

In [ ]:
import praw
import pandas as pd
import time
import gspread
from google.oauth2.service_account import Credentials
import spacy
import datetime
import re
from collections import Counter
import requests
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download required NLTK data
try:
    nltk.download('vader_lexicon', quiet=True)
except:
    pass

# --- Configuration ---
SERVICE_ACCOUNT_FILE = 'YOUR_SERVICE ACCOUNT_FILE'
MOVIE_SHEET_ID = 'YOUR_MOVIE SHEET_ID'
MOVIE_WORKSHEET_NAME = 'movies'
NAME_COLUMN_IN_GSHEET = 'Movie Names'

SUBREDDIT_NAME = 'movies'
POST_LIMIT = 1000  # Increased from 100 to 1000
COMMENTS_PER_POST = 3
OUTPUT_CSV = 'r_movies_1000_posts.csv'  # Fixed filename - no timestamp
MAX_POSTS_TO_SCAN = 3000  # Increased from 500 to 3000 to find 1000 posts with movies

# Reddit API Configuration
REDDIT_CONFIG = {
    'client_id': 'YOUR_CLIENT_ID',
    'client_secret': "YOUR_CLIENT_SECRET",
    'user_agent': 'MovieSentimentBot by /u/anonymous'
}

# Expanded movie database with popular movies, franchises, and series
COMPREHENSIVE_MOVIES = [
    # Marvel Cinematic Universe
    "Iron Man", "Captain America", "Thor", "The Avengers", "Hulk", "Black Widow",
    "Guardians of the Galaxy", "Ant-Man", "Doctor Strange", "Spider-Man", "Black Panther",
    "Captain Marvel", "Avengers Endgame", "Infinity War", "Civil War", "Winter Soldier",
    "Age of Ultron", "Ragnarok", "Homecoming", "Far From Home", "No Way Home",
    "Eternals", "Shang-Chi", "Multiverse of Madness", "Love and Thunder", "Wakanda Forever",

    # DC Universe
    "Batman", "Superman", "Wonder Woman", "Justice League", "Aquaman", "The Flash",
    "Green Lantern", "Suicide Squad", "Birds of Prey", "Joker", "The Batman",
    "Man of Steel", "Batman v Superman", "Dark Knight", "Dark Knight Rises", "Batman Begins",

    # Star Wars
    "Star Wars", "Empire Strikes Back", "Return of the Jedi", "Phantom Menace",
    "Attack of the Clones", "Revenge of the Sith", "Force Awakens", "Last Jedi",
    "Rise of Skywalker", "Rogue One", "Solo", "Mandalorian",

    # Classic/Popular Films
    "The Shawshank Redemption", "The Godfather", "Pulp Fiction", "Forrest Gump",
    "Inception", "The Matrix", "Goodfellas", "Casablanca", "Citizen Kane",
    "Schindler's List", "Lord of the Rings", "Fellowship of the Ring", "Two Towers",
    "Return of the King", "The Hobbit", "Titanic", "Avatar", "Jurassic Park",
    "E.T.", "Jaws", "Back to the Future", "Indiana Jones", "Raiders of the Lost Ark",

    # Horror
    "Halloween", "Friday the 13th", "Nightmare on Elm Street", "The Exorcist",
    "The Shining", "Psycho", "Scream", "Get Out", "Hereditary", "Midsommar",
    "It", "The Conjuring", "Insidious", "Paranormal Activity", "Saw", "Texas Chainsaw Massacre",

    # Action/Thriller
    "John Wick", "Fast and Furious", "Mission Impossible", "Die Hard", "Lethal Weapon",
    "Terminator", "Alien", "Predator", "Rocky", "Rambo", "Mad Max", "Blade Runner",
    "The Bourne Identity", "James Bond", "Casino Royale", "Skyfall", "Spectre",

    # Comedy
    "Anchorman", "Superbad", "Pineapple Express", "Step Brothers", "Talladega Nights",
    "Zoolander", "Meet the Parents", "There's Something About Mary", "Dumb and Dumber",
    "The Hangover", "Bridesmaids", "Ghostbusters", "Groundhog Day", "Big Lebowski",

    # Drama
    "The Departed", "Scarface", "Casino", "Taxi Driver", "Raging Bull",
    "The Wolf of Wall Street", "Django Unchained", "Kill Bill", "Inglourious Basterds",
    "Once Upon a Time in Hollywood", "Parasite", "Nomadland", "Minari", "Sound of Metal",

    # Animated
    "Toy Story", "Finding Nemo", "The Incredibles", "Up", "Wall-E", "Inside Out",
    "Coco", "Moana", "Frozen", "The Lion King", "Beauty and the Beast", "Aladdin",
    "Shrek", "How to Train Your Dragon", "Despicable Me", "Minions", "Ice Age",

    # Recent Popular
    "Top Gun Maverick", "Everything Everywhere All at Once", "The Batman", "Doctor Strange",
    "Thor Love and Thunder", "Black Panther Wakanda Forever", "Avatar Way of Water",
    "Glass Onion", "Knives Out", "Don't Look Up", "Dune", "No Time to Die", "Spider-Man No Way Home"
]

# Movie abbreviations and slang
MOVIE_ABBREVIATIONS = {
    'mcu': 'marvel cinematic universe',
    'dceu': 'dc extended universe',
    'lotr': 'lord of the rings',
    'sw': 'star wars',
    'hp': 'harry potter',
    'gotg': 'guardians of the galaxy',
    'potc': 'pirates of the caribbean',
    'tdk': 'the dark knight',
    'tdkr': 'the dark knight rises',
    'bb': 'batman begins',
    'bvs': 'batman v superman',
    'mos': 'man of steel',
    'jl': 'justice league',
    'ss': 'suicide squad',
    'ww': 'wonder woman',
    'am': 'aquaman',
    'sm': 'spider-man',
    'im': 'iron man',
    'ca': 'captain america',
    'tws': 'the winter soldier',
    'cw': 'civil war',
    'aou': 'age of ultron',
    'tfa': 'the force awakens',
    'tlj': 'the last jedi',
    'tros': 'the rise of skywalker',
    'rotj': 'return of the jedi',
    'esb': 'empire strikes back',
    'anh': 'a new hope',
    'rots': 'revenge of the sith',
    'aotc': 'attack of the clones',
    'tpm': 'the phantom menace',
    'iw': 'infinity war',
    'eg': 'endgame',
    'nwh': 'no way home',
    'ffh': 'far from home',
    'hc': 'homecoming',
    'ragnarok': 'thor ragnarok',
    'bp': 'black panther',
    'cm': 'captain marvel',
    'ds': 'doctor strange',
    'gotg2': 'guardians of the galaxy vol 2',
    'antman': 'ant-man',
    'jp': 'jurassic park',
    'bttf': 'back to the future',
    'rotla': 'raiders of the lost ark',
    'mi': 'mission impossible'
}

# Minimal blacklist - only truly generic words
STRICT_BLACKLIST = {
    'movie', 'movies', 'film', 'films', 'cinema', 'flick', 'picture', 'show',
    'the', 'a', 'an', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with',
    'this', 'that', 'my', 'your', 'his', 'her', 'our', 'their', 'it', 'its'
}

# Movie context indicators - words that suggest movie discussion
MOVIE_CONTEXT_WORDS = {
    'watch', 'watched', 'watching', 'see', 'seen', 'saw', 'love', 'loved', 'hate', 'hated',
    'like', 'liked', 'dislike', 'enjoy', 'enjoyed', 'favorite', 'favourite', 'fav',
    'best', 'worst', 'good', 'bad', 'great', 'amazing', 'terrible', 'awful',
    'recommend', 'recommended', 'suggest', 'review', 'reviewed', 'rating', 'rated',
    'director', 'directed', 'starring', 'stars', 'cast', 'actor', 'actress',
    'sequel', 'prequel', 'remake', 'reboot', 'trailer', 'teaser', 'clip',
    'spoiler', 'spoilers', 'plot', 'story', 'character', 'characters',
    'scene', 'scenes', 'ending', 'beginning', 'climax', 'twist'
}

class EmotionAnalyzer:
    def __init__(self):
        """Initialize emotion analysis tools"""
        try:
            self.sia = SentimentIntensityAnalyzer()
        except:
            print("Warning: VADER sentiment analyzer not available")
            self.sia = None

    def analyze_emotions(self, text):
        """Analyze emotions in text and return top 3 emotions with scores"""
        if not text or not isinstance(text, str):
            return {
                'emotion_1': 'neutral',
                'emotion_1_score': 0.0,
                'emotion_2': 'neutral',
                'emotion_2_score': 0.0,
                'emotion_3': 'neutral',
                'emotion_3_score': 0.0
            }

        # Clean text for analysis
        text = text.lower().strip()

        # Get basic sentiment using TextBlob
        try:
            blob = TextBlob(text)
            polarity = blob.sentiment.polarity  # -1 to 1
            subjectivity = blob.sentiment.subjectivity  # 0 to 1
        except:
            polarity = 0.0
            subjectivity = 0.0

        # Get VADER sentiment scores
        if self.sia:
            try:
                vader_scores = self.sia.polarity_scores(text)
                compound = vader_scores['compound']
                pos_score = vader_scores['pos']
                neg_score = vader_scores['neg']
                neu_score = vader_scores['neu']
            except:
                compound = pos_score = neg_score = neu_score = 0.0
        else:
            compound = pos_score = neg_score = neu_score = 0.0

        # Emotion keyword detection with scores
        emotions = self._detect_emotion_keywords(text)

        # Combine different approaches to determine top emotions
        emotion_scores = {
            'joy': max(pos_score, polarity if polarity > 0 else 0) + emotions.get('joy', 0),
            'excitement': emotions.get('excitement', 0) + (pos_score * 0.8),
            'love': emotions.get('love', 0) + (pos_score * 0.6),
            'anger': neg_score + emotions.get('anger', 0),
            'disappointment': emotions.get('disappointment', 0) + (neg_score * 0.7),
            'fear': emotions.get('fear', 0) + (neg_score * 0.5),
            'sadness': emotions.get('sadness', 0) + (neg_score * 0.6),
            'surprise': emotions.get('surprise', 0) + (abs(polarity) * subjectivity),
            'nostalgia': emotions.get('nostalgia', 0),
            'confusion': emotions.get('confusion', 0) + (subjectivity * 0.5),
            'anticipation': emotions.get('anticipation', 0),
            'neutral': neu_score
        }

        # Get top 3 emotions
        sorted_emotions = sorted(emotion_scores.items(), key=lambda x: x[1], reverse=True)
        top_3 = sorted_emotions[:3]

        # Ensure we have 3 emotions
        while len(top_3) < 3:
            top_3.append(('neutral', 0.0))

        return {
            'emotion_1': top_3[0][0],
            'emotion_1_score': round(top_3[0][1], 3),
            'emotion_2': top_3[1][0],
            'emotion_2_score': round(top_3[1][1], 3),
            'emotion_3': top_3[2][0],
            'emotion_3_score': round(top_3[2][1], 3)
        }

    def _detect_emotion_keywords(self, text):
        """Detect emotion-specific keywords and return scores"""
        emotion_keywords = {
            'joy': ['amazing', 'awesome', 'fantastic', 'incredible', 'brilliant', 'outstanding',
                   'excellent', 'wonderful', 'perfect', 'beautiful', 'hilarious', 'funny'],
            'excitement': ['excited', 'thrilled', 'pumped', 'hyped', 'anticipating', 'cant wait',
                          'looking forward', 'epic', 'incredible', 'mind-blowing'],
            'love': ['love', 'adore', 'favorite', 'favourite', 'obsessed', 'amazing chemistry',
                    'perfect cast', 'masterpiece', 'classic'],
            'anger': ['hate', 'terrible', 'awful', 'worst', 'horrible', 'disgusting', 'garbage',
                     'trash', 'ruined', 'destroyed', 'furious', 'angry'],
            'disappointment': ['disappointed', 'letdown', 'expected better', 'underwhelming',
                              'waste of time', 'boring', 'meh', 'overrated', 'overhyped'],
            'fear': ['scared', 'terrifying', 'creepy', 'disturbing', 'nightmare', 'horrifying',
                    'spine-chilling', 'jump scare', 'suspenseful'],
            'sadness': ['sad', 'depressing', 'heartbreaking', 'emotional', 'tear jerker', 'cried',
                       'touching', 'moving', 'tragic'],
            'surprise': ['shocked', 'unexpected', 'plot twist', 'surprised', 'didnt see coming',
                        'jaw dropped', 'mind blown', 'revelation'],
            'nostalgia': ['nostalgic', 'childhood', 'memories', 'throwback', 'classic', 'old school',
                         'brings back', 'remember when'],
            'confusion': ['confused', 'wtf', 'what the', 'makes no sense', 'plot holes',
                         'convoluted', 'complicated', 'lost me'],
            'anticipation': ['cant wait', 'excited for', 'looking forward', 'hoping', 'sequel',
                           'part 2', 'next movie', 'continuation']
        }

        emotion_scores = {}
        words = text.split()

        for emotion, keywords in emotion_keywords.items():
            score = 0
            for keyword in keywords:
                # Count occurrences of keywords
                if keyword in text:
                    score += text.count(keyword) * 0.2

                # Check for partial matches in words
                for word in words:
                    if keyword in word:
                        score += 0.1

            emotion_scores[emotion] = min(score, 1.0)  # Cap at 1.0

        return emotion_scores

class AdvancedMovieDetector:
    def __init__(self):
        print("Initializing enhanced movie detection system...")
        # Load spaCy with only essential components
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
        self.movie_titles = set()
        self.movie_patterns = []
        self.setup_detection_system()

    def setup_detection_system(self):
        """Setup comprehensive movie detection"""
        # Load all movies
        all_movies = COMPREHENSIVE_MOVIES.copy()

        # Try to load from Google Sheets if available
        try:
            additional_movies = self._load_from_google_sheets()
            if additional_movies:
                all_movies.extend(additional_movies)
        except:
            pass

        # Process all movie titles
        self._process_all_movies(all_movies)

        # Setup EntityRuler with aggressive patterns
        self._setup_entity_ruler()

        print(f"Detection system ready with {len(self.movie_titles)} movie patterns")

    def _load_from_google_sheets(self):
        """Try to load additional movies from Google Sheets"""
        try:
            import os
            if not os.path.exists(SERVICE_ACCOUNT_FILE):
                return None

            scopes = ['https://www.googleapis.com/auth/spreadsheets']
            creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scopes)
            client = gspread.authorize(creds)
            sheet = client.open_by_key(MOVIE_SHEET_ID).worksheet(MOVIE_WORKSHEET_NAME)
            all_data = sheet.get_all_records()
            df = pd.DataFrame(all_data)

            if NAME_COLUMN_IN_GSHEET in df.columns:
                movies = df[NAME_COLUMN_IN_GSHEET].dropna().astype(str).tolist()
                print(f"Loaded {len(movies)} additional movies from Google Sheets")
                return movies
        except Exception as e:
            print(f"Could not load from Google Sheets: {e}")
        return None

    def _process_all_movies(self, movies):
        """Process and clean all movie titles"""
        for movie in movies:
            if not movie or len(str(movie).strip()) < 2:
                continue

            movie = str(movie).strip()

            # Add original title
            cleaned = self._clean_title(movie)
            if cleaned and self._is_valid_movie_title(cleaned):
                self.movie_titles.add(cleaned.lower())

                # Add variations
                self._add_variations(cleaned)

        # Add abbreviations
        for abbr, full_name in MOVIE_ABBREVIATIONS.items():
            self.movie_titles.add(abbr.lower())
            self.movie_titles.add(full_name.lower())

    def _clean_title(self, title):
        """Clean movie title"""
        # Remove year in parentheses
        title = re.sub(r'\s*\(\d{4}\).*$', '', title)

        # Remove common suffixes
        title = re.sub(r'\s+(movie|film)$', '', title, flags=re.IGNORECASE)

        # Clean special characters but keep apostrophes and hyphens
        title = re.sub(r'[^\w\s\'\-]', ' ', title)

        # Normalize whitespace
        title = ' '.join(title.split())

        return title.strip()

    def _is_valid_movie_title(self, title):
        """Check if title is valid - very permissive now"""
        title_lower = title.lower().strip()

        # Only reject if it's in the strict blacklist
        if title_lower in STRICT_BLACKLIST:
            return False

        # Must be at least 2 characters
        if len(title_lower) < 2:
            return False

        return True

    def _add_variations(self, title):
        """Add common variations"""
        title_lower = title.lower()

        # Add version without "the"
        if title_lower.startswith('the '):
            self.movie_titles.add(title_lower[4:])

        # Add version without apostrophes
        if "'" in title_lower:
            self.movie_titles.add(title_lower.replace("'", ""))
            self.movie_titles.add(title_lower.replace("'", ""))

        # Add hyphenated version
        if ' ' in title_lower and len(title_lower.split()) == 2:
            self.movie_titles.add(title_lower.replace(' ', '-'))

    def _setup_entity_ruler(self):
        """Setup EntityRuler with comprehensive patterns"""
        ruler = self.nlp.add_pipe("entity_ruler", before="ner")

        patterns = []
        for movie in self.movie_titles:
            patterns.append({"label": "MOVIE", "pattern": movie})

        ruler.add_patterns(patterns)
        print(f"Added {len(patterns)} patterns to EntityRuler")

    def detect_movies_comprehensive(self, text):
        """Comprehensive movie detection using multiple strategies"""
        if not text or not isinstance(text, str):
            return []

        text = text.lower().strip()
        found_movies = set()

        # Strategy 1: spaCy EntityRuler
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ == "MOVIE":
                movie = ent.text.strip()
                if movie in MOVIE_ABBREVIATIONS:
                    movie = MOVIE_ABBREVIATIONS[movie]
                found_movies.add(movie.title())

        # Strategy 2: Direct string matching with context
        found_movies.update(self._contextual_string_matching(text))

        # Strategy 3: Quoted text detection
        found_movies.update(self._detect_quoted_titles(text))

        # Strategy 4: Capitalized sequences
        found_movies.update(self._detect_capitalized_sequences(text))

        # Filter and validate results
        validated_movies = []
        for movie in found_movies:
            if self._validate_detection(movie, text):
                validated_movies.append(movie)

        return list(set(validated_movies))  # Remove duplicates

    def _contextual_string_matching(self, text):
        """Find movies using context-aware string matching"""
        found = set()
        words = text.split()

        for i, word in enumerate(words):
            # Check if current word or phrase matches a movie
            for length in range(1, min(6, len(words) - i + 1)):  # Check up to 5-word phrases
                phrase = ' '.join(words[i:i+length])

                if phrase in self.movie_titles:
                    # Check for movie context nearby
                    context_start = max(0, i - 5)
                    context_end = min(len(words), i + length + 5)
                    context = ' '.join(words[context_start:context_end])

                    if any(ctx_word in context for ctx_word in MOVIE_CONTEXT_WORDS):
                        # Expand abbreviations
                        if phrase in MOVIE_ABBREVIATIONS:
                            phrase = MOVIE_ABBREVIATIONS[phrase]
                        found.add(phrase.title())

        return found

    def _detect_quoted_titles(self, text):
        """Detect movie titles in quotes"""
        found = set()

        # Find text in quotes
        quote_patterns = [
            r'"([^"]{2,50})"',  # Double quotes
            r"'([^']{2,50})'",  # Single quotes
            r'\"([^\"]{2,50})\"'  # Escaped quotes
        ]

        for pattern in quote_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if match.lower().strip() in self.movie_titles:
                    found.add(match.strip().title())

        return found

    def _detect_capitalized_sequences(self, text):
        """Detect capitalized word sequences that might be movie titles"""
        found = set()

        # Find sequences of capitalized words
        cap_pattern = r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b'
        matches = re.findall(cap_pattern, text)

        for match in matches:
            if match.lower() in self.movie_titles:
                found.add(match.title())

        return found

    def _validate_detection(self, movie, full_text):
        """Validate movie detection - very permissive"""
        if not movie or len(movie.strip()) < 2:
            return False

        movie_lower = movie.lower().strip()

        # Reject only strict blacklist items
        if movie_lower in STRICT_BLACKLIST:
            return False

        # Accept if it's a known abbreviation
        if movie_lower in MOVIE_ABBREVIATIONS:
            return True

        # Accept if it's in our movie database
        if movie_lower in self.movie_titles:
            return True

        # Accept multi-word titles (more likely to be specific)
        if len(movie.split()) > 1:
            return True

        return False

class RedditScraper:
    def __init__(self, reddit_config):
        self.reddit = praw.Reddit(**reddit_config)
        self.emotion_analyzer = EmotionAnalyzer()

    def collect_data(self, subreddit_name, post_limit, comments_per_post, movie_detector, max_posts_to_scan):
        """Collect Reddit data with enhanced movie detection and emotion analysis - skip posts without movies"""
        all_data = []
        posts_with_movies = 0
        posts_scanned = 0
        skipped_posts = 0

        print(f"\nScraping {post_limit} posts WITH movies from r/{subreddit_name}...")
        print(f"Will scan up to {max_posts_to_scan} posts to find {post_limit} with detected movies")

        try:
            subreddit = self.reddit.subreddit(subreddit_name)

            # Mix different sorting methods to get more diverse posts
            post_sources = [
                ('hot', max_posts_to_scan // 3),
                ('new', max_posts_to_scan // 3),
                ('top', max_posts_to_scan // 3)
            ]

            for sort_method, limit in post_sources:
                if posts_with_movies >= post_limit:
                    break

                print(f"\nFetching {sort_method} posts...")

                if sort_method == 'hot':
                    posts = subreddit.hot(limit=limit)
                elif sort_method == 'new':
                    posts = subreddit.new(limit=limit)
                elif sort_method == 'top':
                    posts = subreddit.top(time_filter='week', limit=limit)

                for submission in posts:
                    posts_scanned += 1

                    # Check if we've found enough posts with movies
                    if posts_with_movies >= post_limit:
                        break

                    # Combine title and content for analysis
                    post_text = f"{submission.title}. {submission.selftext}".strip()

                    # Detect movies in post
                    found_movies = movie_detector.detect_movies_comprehensive(post_text)

                    # Also check comments for additional movies
                    comment_text = ""
                    comment_movies = []

                    if submission.num_comments > 0:
                        try:
                            submission.comments.replace_more(limit=0)
                            comments = [c.body for c in submission.comments[:comments_per_post]
                                      if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']]
                            comment_text = " ".join(comments[:3])  # Limit comment text

                            if comment_text:
                                comment_movies = movie_detector.detect_movies_comprehensive(comment_text)

                        except Exception as e:
                            print(f"Error processing comments for post {posts_scanned}: {e}")

                    # Combine all found movies
                    all_found_movies = list(set(found_movies + comment_movies))

                    # Skip this post if no movies were detected
                    if not all_found_movies:
                        skipped_posts += 1
                        if posts_scanned % 50 == 0:
                            print(f"Scanned {posts_scanned} posts | Found {posts_with_movies}/{post_limit} with movies | Skipped {skipped_posts}")
                        continue

                    # We found movies! Process this post
                    posts_with_movies += 1

                    # Analyze emotions in the combined text
                    full_text_for_emotion = f"{post_text} {comment_text}"
                    emotion_results = self.emotion_analyzer.analyze_emotions(full_text_for_emotion)

                    # Store results (no 'Unknown' movies anymore)
                    data_entry = {
                        'Title': submission.title,
                        'Author': str(submission.author) if submission.author else '[deleted]',
                        'All_Movies': all_found_movies,
                        'Primary_Movie': all_found_movies[0],  # Always has at least one movie
                        'Movie_Count': len(all_found_movies),
                        'Content': submission.selftext[:200] + '...' if len(submission.selftext) > 200 else submission.selftext,
                        'Comments_Sample': comment_text[:300] + '...' if len(comment_text) > 300 else comment_text,
                        'Timestamp_UTC': datetime.datetime.utcfromtimestamp(submission.created_utc),
                        'Sort_Method': sort_method
                    }

                    # Add emotion analysis results
                    data_entry.update(emotion_results)
                    all_data.append(data_entry)

                    if posts_with_movies % 25 == 0:
                        print(f"Progress: {posts_with_movies}/{post_limit} posts with movies | Scanned {posts_scanned} total | Skipped {skipped_posts}")

                    time.sleep(0.15)  # Slightly reduced rate limiting for efficiency

        except Exception as e:
            print(f"Error during scraping: {e}")

        print(f"\nScraping complete!")
        print(f"Posts scanned: {posts_scanned}")
        print(f"Posts with movies found: {posts_with_movies}")
        print(f"Posts skipped (no movies): {skipped_posts}")
        print(f"Success rate: {posts_with_movies/posts_scanned*100:.1f}%")

        return pd.DataFrame(all_data)

def main():
    """Main execution function"""
    try:
        # Initialize enhanced movie detector
        print("=== Reddit Movie Sentiment Analysis - 1000 Posts ===")
        detector = AdvancedMovieDetector()

        # Initialize Reddit scraper
        scraper = RedditScraper(REDDIT_CONFIG)

        # Collect data - only posts with detected movies
        df = scraper.collect_data(SUBREDDIT_NAME, POST_LIMIT, COMMENTS_PER_POST, detector, MAX_POSTS_TO_SCAN)

        # Save results with fixed filename
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\nData saved to {OUTPUT_CSV}")

        # Show comprehensive results
        print(f"\n=== DETECTION RESULTS ===")
        print(f"Total posts with movies collected: {len(df)}")

        if len(df) > 0:
            # Movie statistics - all posts now have movies
            all_movies = []
            for movies_list in df['All_Movies']:
                if isinstance(movies_list, list):
                    all_movies.extend(movies_list)

            if all_movies:
                movie_counts = Counter(all_movies)
                print(f"\nTop 20 detected movies:")
                for movie, count in movie_counts.most_common(20):
                    print(f"  {movie}: {count}")

            # Show distribution by sort method
            if 'Sort_Method' in df.columns:
                sort_distribution = df['Sort_Method'].value_counts()
                print(f"\nPosts by source:")
                for method, count in sort_distribution.items():
                    print(f"  {method}: {count}")

            # Show sample detections with emotions
            print(f"\nSample movie detections with emotions:")
            sample = df[['Title', 'Primary_Movie', 'Movie_Count', 'emotion_1', 'emotion_1_score']].head(15)
            for idx, row in sample.iterrows():
                title_short = row['Title'][:60] + '...' if len(row['Title']) > 60 else row['Title']
                print(f"  '{title_short}' -> {row['Primary_Movie']} | {row['emotion_1']} ({row['emotion_1_score']:.3f})")

            # Show emotion statistics
            print(f"\nTop emotions detected:")
            all_emotions = list(df['emotion_1']) + list(df['emotion_2']) + list(df['emotion_3'])
            emotion_counts = Counter(all_emotions)
            for emotion, count in emotion_counts.most_common(12):
                print(f"  {emotion}: {count}")

            # Show posts with multiple movies
            multi_movie_df = df[df['Movie_Count'] > 1]
            print(f"\nPosts with multiple movies detected: {len(multi_movie_df)}")
            print(f"Average movies per post: {df['Movie_Count'].mean():.2f}")
            print(f"Maximum movies in single post: {df['Movie_Count'].max()}")

            # Show time distribution
            if 'Timestamp_UTC' in df.columns:
                df['Hour'] = pd.to_datetime(df['Timestamp_UTC']).dt.hour
                hour_dist = df['Hour'].value_counts().head(5)
                print(f"\nTop posting hours (UTC):")
                for hour, count in hour_dist.items():
                    print(f"  {hour}:00 - {count} posts")

            # Quality metrics
            avg_content_length = df['Content'].str.len().mean()
            posts_with_comments = len(df[df['Comments_Sample'].str.len() > 10])

            print(f"\n=== QUALITY METRICS ===")
            print(f"Average content length: {avg_content_length:.0f} characters")
            print(f"Posts with meaningful comments: {posts_with_comments}")
            print(f"Unique movies detected: {len(set(all_movies))}")

        print(f"\n=== ANALYSIS COMPLETE ===")
        print(f"Dataset contains {len(df)} posts, all with detected movies")
        print(f"Data includes movie detection + top 3 emotions with scores for each post")
        print(f"Mixed data sources (hot/new/top) for better diversity")
        print(f"No 'Unknown' entries - only posts with successfully detected movies are included")
        print(f"Data saved to: {OUTPUT_CSV}")

        # Additional analysis suggestions
        if len(df) >= 1000:
            print(f"\n=== SUCCESS! ===")
            print(f"✓ Successfully collected {len(df)} posts with movie mentions")
            print(f"✓ Ready for sentiment analysis and machine learning")
            print(f"✓ Comprehensive emotion data available")
            print(f"✓ Data saved to: {OUTPUT_CSV}")
        else:
            print(f"\n=== PARTIAL SUCCESS ===")
            print(f"⚠ Collected {len(df)} posts (target was 1000)")
            print(f"⚠ Consider running again or adjusting detection parameters")
            print(f"⚠ Data saved to: {OUTPUT_CSV}")

    except Exception as e:
        print(f"Script failed: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'praw'

### Visualization

In [ ]:
import pandas as pd
from collections import Counter
import ast

# Load the CSV
df = pd.read_csv("r_movies_1000_posts.csv")

# Convert each "All_Movies" entry to a list
def convert_to_list(cell):
    if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == '[]':
        return []
    try:
        # Use ast.literal_eval to safely parse the string representation of a list
        return [movie.strip().lower() for movie in ast.literal_eval(cell)]
    except (ValueError, SyntaxError):
        print(f"Warning: Could not parse '{cell}'. Skipping this row.")
        return []

df["All_Movies"] = df["All_Movies"].apply(convert_to_list)

# Count and sort movie mentions
movie_counter = Counter()
for movie_list in df["All_Movies"]:
    movie_counter.update(movie_list)

# Sort by frequency (descending)
movie_counts = dict(sorted(movie_counter.items(), key=lambda item: item[1], reverse=True))

# Print each movie and its count
for movie, count in movie_counts.items():
    print(f"{movie}: {count}")

up: 484
captain america: 239
saw: 182
aquaman: 164
star wars: 42
iron man: 36
alien: 35
lord of the rings: 32
jurassic park: 29
terminator: 26
superman: 23
rocky: 20
the godfather: 20
the matrix: 20
mission impossible: 19
spider-man: 18
get out: 18
jaws: 16
mad max: 16
godfather: 15
the dark knight: 15
avengers: 15
predator: 14
the lion king: 14
joker: 14
batman: 14
scream: 14
the shawshank redemption: 13
avatar: 13
inception: 13
pulp fiction: 12
blade runner: 12
once upon a time in hollywood: 12
casablanca: 12
big lebowski: 11
dune: 11
fellowship of the ring: 11
the exorcist: 11
midsommar: 11
psycho: 11
back to the future: 10
matrix: 10
the departed: 10
die hard: 9
solo: 9
titanic: 9
parasite: 9
harry potter: 9
lethal weapon: 8
james bond: 8
taxi driver: 8
inside out: 8
indiana jones: 8
raiders of the lost ark: 8
everything everywhere all at once: 7
halloween: 7
endgame: 7
the shining: 6
the conjuring: 6
shawshank redemption: 6
frozen: 6
texas chainsaw massacre: 6
hereditary: 6
toy st

**Top 10 movies r/movies**

In [ ]:
import matplotlib.pyplot as plt

# Choose how many top movies to show
TOP_N = 10

# Get top N items
top_movies = list(movie_counts.items())[:TOP_N]
movie_names, counts = zip(*top_movies)

# Plotting
plt.figure(figsize=(12, 6))
plt.barh(movie_names[::-1], counts[::-1], color='lavender')  # reverse for descending top-down
plt.xlabel("Mention Count")
plt.title(f"Top {TOP_N} Most Mentioned Movies in r/movies")
plt.tight_layout()
plt.show()

**Average Sentiment Scores per Day r/movies**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Load the data
df = pd.read_csv("r_movies_1000_posts.csv")

# Extract date from 'Timestamp_UTC'
df['Date'] = pd.to_datetime(df['Timestamp_UTC'], errors='coerce').dt.date

# Parse emotions from individual columns (emotion_1, emotion_2, emotion_3 with their scores)
def parse_emotions_from_columns(row):
    emotions = {}
    # Get emotion scores from the separate columns
    emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']
    score_columns = ['emotion_1_score', 'emotion_2_score', 'emotion_3_score']

    for i, (emo_col, score_col) in enumerate(zip(emotion_columns, score_columns)):
        if emo_col in row and score_col in row:
            if pd.notna(row[emo_col]) and pd.notna(row[score_col]):
                emotion = str(row[emo_col]).lower()
                score = float(row[score_col])
                emotions[emotion] = score
    return emotions

# Apply parsing and expand to new columns
emotion_df = df.apply(parse_emotions_from_columns, axis=1).apply(pd.Series)
emotion_df['Date'] = df['Date']

# Group by date and average
daily_sentiments = emotion_df.groupby('Date').mean().reset_index()

# Melt for seaborn
melted = daily_sentiments.melt(id_vars='Date', var_name='Emotion', value_name='Score')

# Plot
plt.figure(figsize=(16, 6))
sns.lineplot(data=melted, x='Date', y='Score', hue='Emotion', marker='o')
plt.title("Average Sentiment Scores per Day in r/movies", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Average Sentiment Score", fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Sentiment", fontsize=10, title_fontsize=12)
plt.tight_layout()
plt.show()

# **Final Code r/letterboxd**

### CODE

In [ ]:
import praw
import pandas as pd
import time
import gspread
from google.oauth2.service_account import Credentials
import spacy
import datetime
import re
from collections import Counter
import requests
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download required NLTK data
try:
    nltk.download('vader_lexicon', quiet=True)
except:
    pass

# --- Configuration ---
SERVICE_ACCOUNT_FILE = 'YOUR_SERVICE_ACCOUNT_FILE'
MOVIE_SHEET_ID = 'YOUR_MOVIE_SHEET_ID'
MOVIE_WORKSHEET_NAME = 'movies'
NAME_COLUMN_IN_GSHEET = 'Movie Names'

SUBREDDIT_NAME = 'letterboxd'
POST_LIMIT = 1000
COMMENTS_PER_POST = 3
OUTPUT_CSV = 'r_letterboxd_1000_posts.csv'  # Fixed filename as requested
MAX_POSTS_TO_SCAN = 3000

# Reddit API Configuration
REDDIT_CONFIG = {
    'client_id': 'YOUR_CLIENT_ID',
    'client_secret': "YOUR_CLIENT_SECRET",
    'user_agent': 'LetterboxdSentimentBot by /u/anonymous'
}

# Expanded movie database with popular movies, franchises, and series
COMPREHENSIVE_MOVIES = [
    # Marvel Cinematic Universe
    "Iron Man", "Captain America", "Thor", "The Avengers", "Hulk", "Black Widow",
    "Guardians of the Galaxy", "Ant-Man", "Doctor Strange", "Spider-Man", "Black Panther",
    "Captain Marvel", "Avengers Endgame", "Infinity War", "Civil War", "Winter Soldier",
    "Age of Ultron", "Ragnarok", "Homecoming", "Far From Home", "No Way Home",
    "Eternals", "Shang-Chi", "Multiverse of Madness", "Love and Thunder", "Wakanda Forever",

    # DC Universe
    "Batman", "Superman", "Wonder Woman", "Justice League", "Aquaman", "The Flash",
    "Green Lantern", "Suicide Squad", "Birds of Prey", "Joker", "The Batman",
    "Man of Steel", "Batman v Superman", "Dark Knight", "Dark Knight Rises", "Batman Begins",

    # Star Wars
    "Star Wars", "Empire Strikes Back", "Return of the Jedi", "Phantom Menace",
    "Attack of the Clones", "Revenge of the Sith", "Force Awakens", "Last Jedi",
    "Rise of Skywalker", "Rogue One", "Solo", "Mandalorian",

    # Classic/Popular Films
    "The Shawshank Redemption", "The Godfather", "Pulp Fiction", "Forrest Gump",
    "Inception", "The Matrix", "Goodfellas", "Casablanca", "Citizen Kane",
    "Schindler's List", "Lord of the Rings", "Fellowship of the Ring", "Two Towers",
    "Return of the King", "The Hobbit", "Titanic", "Avatar", "Jurassic Park",
    "E.T.", "Jaws", "Back to the Future", "Indiana Jones", "Raiders of the Lost Ark",

    # Horror
    "Halloween", "Friday the 13th", "Nightmare on Elm Street", "The Exorcist",
    "The Shining", "Psycho", "Scream", "Get Out", "Hereditary", "Midsommar",
    "It", "The Conjuring", "Insidious", "Paranormal Activity", "Saw", "Texas Chainsaw Massacre",

    # Action/Thriller
    "John Wick", "Fast and Furious", "Mission Impossible", "Die Hard", "Lethal Weapon",
    "Terminator", "Alien", "Predator", "Rocky", "Rambo", "Mad Max", "Blade Runner",
    "The Bourne Identity", "James Bond", "Casino Royale", "Skyfall", "Spectre",

    # Comedy
    "Anchorman", "Superbad", "Pineapple Express", "Step Brothers", "Talladega Nights",
    "Zoolander", "Meet the Parents", "There's Something About Mary", "Dumb and Dumber",
    "The Hangover", "Bridesmaids", "Ghostbusters", "Groundhog Day", "Big Lebowski",

    # Drama
    "The Departed", "Scarface", "Casino", "Taxi Driver", "Raging Bull",
    "The Wolf of Wall Street", "Django Unchained", "Kill Bill", "Inglourious Basterds",
    "Once Upon a Time in Hollywood", "Parasite", "Nomadland", "Minari", "Sound of Metal",

    # Animated
    "Toy Story", "Finding Nemo", "The Incredibles", "Up", "Wall-E", "Inside Out",
    "Coco", "Moana", "Frozen", "The Lion King", "Beauty and the Beast", "Aladdin",
    "Shrek", "How to Train Your Dragon", "Despicable Me", "Minions", "Ice Age",

    # Recent Popular
    "Top Gun Maverick", "Everything Everywhere All at Once", "The Batman", "Doctor Strange",
    "Thor Love and Thunder", "Black Panther Wakanda Forever", "Avatar Way of Water",
    "Glass Onion", "Knives Out", "Don't Look Up", "Dune", "No Time to Die", "Spider-Man No Way Home",

    # Letterboxd Popular/Art House Films
    "Mulholland Drive", "2001 A Space Odyssey", "Vertigo", "8½", "Persona", "Tokyo Story",
    "The Rules of the Game", "Bicycle Thieves", "Singin' in the Rain", "Some Like It Hot",
    "North by Northwest", "Psycho", "La Dolce Vita", "The 400 Blows", "Breathless",
    "Contempt", "Pierrot le Fou", "Cries and Whispers", "Scenes from a Marriage",
    "Annie Hall", "Manhattan", "Taxi Driver", "Raging Bull", "GoodFellas",
    "The Seventh Seal", "Wild Strawberries", "Fanny and Alexander", "Amour",
    "Blue Is the Warmest Color", "Call Me by Your Name", "Moonlight", "Lady Bird",
    "Portrait of a Lady on Fire", "The Handmaiden", "Burning", "Shoplifters",
    "Roma", "The Favourite", "Marriage Story", "Phantom Thread", "There Will Be Blood"
]

# Movie abbreviations and slang
MOVIE_ABBREVIATIONS = {
    'mcu': 'marvel cinematic universe',
    'dceu': 'dc extended universe',
    'lotr': 'lord of the rings',
    'sw': 'star wars',
    'hp': 'harry potter',
    'gotg': 'guardians of the galaxy',
    'potc': 'pirates of the caribbean',
    'tdk': 'the dark knight',
    'tdkr': 'the dark knight rises',
    'bb': 'batman begins',
    'bvs': 'batman v superman',
    'mos': 'man of steel',
    'jl': 'justice league',
    'ss': 'suicide squad',
    'ww': 'wonder woman',
    'am': 'aquaman',
    'sm': 'spider-man',
    'im': 'iron man',
    'ca': 'captain america',
    'tws': 'the winter soldier',
    'cw': 'civil war',
    'aou': 'age of ultron',
    'tfa': 'the force awakens',
    'tlj': 'the last jedi',
    'tros': 'the rise of skywalker',
    'rotj': 'return of the jedi',
    'esb': 'empire strikes back',
    'anh': 'a new hope',
    'rots': 'revenge of the sith',
    'aotc': 'attack of the clones',
    'tpm': 'the phantom menace',
    'iw': 'infinity war',
    'eg': 'endgame',
    'nwh': 'no way home',
    'ffh': 'far from home',
    'hc': 'homecoming',
    'ragnarok': 'thor ragnarok',
    'bp': 'black panther',
    'cm': 'captain marvel',
    'ds': 'doctor strange',
    'gotg2': 'guardians of the galaxy vol 2',
    'antman': 'ant-man',
    'jp': 'jurassic park',
    'bttf': 'back to the future',
    'rotla': 'raiders of the lost ark',
    'mi': 'mission impossible',
    'lbd': 'letterboxd',
    'lb': 'letterboxd',
    'potlaofl': 'portrait of a lady on fire',
    'eeoao': 'everything everywhere all at once',
    'twbb': 'there will be blood',
    'md': 'mulholland drive',
    '2001': '2001 a space odyssey'
}

# Minimal blacklist
STRICT_BLACKLIST = {
    'movie', 'movies', 'film', 'films', 'cinema', 'flick', 'picture', 'show',
    'the', 'a', 'an', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with',
    'this', 'that', 'my', 'your', 'his', 'her', 'our', 'their', 'it', 'its',
    'letterboxd', 'lb', 'lbd', 'diary', 'review', 'reviews', 'rating', 'ratings'
}

# Movie context indicators
MOVIE_CONTEXT_WORDS = {
    'watch', 'watched', 'watching', 'see', 'seen', 'saw', 'love', 'loved', 'hate', 'hated',
    'like', 'liked', 'dislike', 'enjoy', 'enjoyed', 'favorite', 'favourite', 'fav',
    'best', 'worst', 'good', 'bad', 'great', 'amazing', 'terrible', 'awful',
    'recommend', 'recommended', 'suggest', 'review', 'reviewed', 'rating', 'rated',
    'director', 'directed', 'starring', 'stars', 'cast', 'actor', 'actress',
    'sequel', 'prequel', 'remake', 'reboot', 'trailer', 'teaser', 'clip',
    'spoiler', 'spoilers', 'plot', 'story', 'character', 'characters',
    'scene', 'scenes', 'ending', 'beginning', 'climax', 'twist',
    'diary', 'log', 'logged', 'list', 'lists', 'four-five', 'half-star', 'stars',
    'rewatch', 'rewatched', 'rewatching', 'first-time', 'blind-watch', 'criterion',
    'arthouse', 'art-house', 'foreign', 'subtitled', 'masterpiece', 'classic',
    'underrated', 'overrated', 'auteur', 'cinephile', 'filmmaking', 'cinematography',
    'screenplay', 'score', 'soundtrack', 'editing', 'production', 'festival'
}

class EmotionAnalyzer:
    def __init__(self):
        """Initialize emotion analysis tools"""
        try:
            self.sia = SentimentIntensityAnalyzer()
        except:
            print("Warning: VADER sentiment analyzer not available")
            self.sia = None

    def analyze_emotions(self, text):
        """Analyze emotions in text and return top 3 emotions with scores"""
        if not text or not isinstance(text, str):
            return {
                'emotion_1': 'neutral',
                'emotion_1_score': 0.0,
                'emotion_2': 'neutral',
                'emotion_2_score': 0.0,
                'emotion_3': 'neutral',
                'emotion_3_score': 0.0
            }

        # Clean text for analysis
        text = text.lower().strip()

        # Get basic sentiment using TextBlob
        try:
            blob = TextBlob(text)
            polarity = blob.sentiment.polarity
            subjectivity = blob.sentiment.subjectivity
        except:
            polarity = 0.0
            subjectivity = 0.0

        # Get VADER sentiment scores
        if self.sia:
            try:
                vader_scores = self.sia.polarity_scores(text)
                compound = vader_scores['compound']
                pos_score = vader_scores['pos']
                neg_score = vader_scores['neg']
                neu_score = vader_scores['neu']
            except:
                compound = pos_score = neg_score = neu_score = 0.0
        else:
            compound = pos_score = neg_score = neu_score = 0.0

        # Emotion keyword detection with scores
        emotions = self._detect_emotion_keywords(text)

        # Combine different approaches to determine top emotions
        emotion_scores = {
            'joy': max(pos_score, polarity if polarity > 0 else 0) + emotions.get('joy', 0),
            'excitement': emotions.get('excitement', 0) + (pos_score * 0.8),
            'love': emotions.get('love', 0) + (pos_score * 0.6),
            'anger': neg_score + emotions.get('anger', 0),
            'disappointment': emotions.get('disappointment', 0) + (neg_score * 0.7),
            'fear': emotions.get('fear', 0) + (neg_score * 0.5),
            'sadness': emotions.get('sadness', 0) + (neg_score * 0.6),
            'surprise': emotions.get('surprise', 0) + (abs(polarity) * subjectivity),
            'nostalgia': emotions.get('nostalgia', 0),
            'confusion': emotions.get('confusion', 0) + (subjectivity * 0.5),
            'anticipation': emotions.get('anticipation', 0),
            'appreciation': emotions.get('appreciation', 0),
            'neutral': neu_score
        }

        # Get top 3 emotions
        sorted_emotions = sorted(emotion_scores.items(), key=lambda x: x[1], reverse=True)
        top_3 = sorted_emotions[:3]

        # Ensure we have 3 emotions
        while len(top_3) < 3:
            top_3.append(('neutral', 0.0))

        return {
            'emotion_1': top_3[0][0],
            'emotion_1_score': round(top_3[0][1], 3),
            'emotion_2': top_3[1][0],
            'emotion_2_score': round(top_3[1][1], 3),
            'emotion_3': top_3[2][0],
            'emotion_3_score': round(top_3[2][1], 3)
        }

    def _detect_emotion_keywords(self, text):
        """Detect emotion-specific keywords and return scores"""
        emotion_keywords = {
            'joy': ['amazing', 'awesome', 'fantastic', 'incredible', 'brilliant', 'outstanding',
                   'excellent', 'wonderful', 'perfect', 'beautiful', 'hilarious', 'funny'],
            'excitement': ['excited', 'thrilled', 'pumped', 'hyped', 'anticipating', 'cant wait',
                          'looking forward', 'epic', 'incredible', 'mind-blowing'],
            'love': ['love', 'adore', 'favorite', 'favourite', 'obsessed', 'amazing chemistry',
                    'perfect cast', 'masterpiece', 'classic'],
            'anger': ['hate', 'terrible', 'awful', 'worst', 'horrible', 'disgusting', 'garbage',
                     'trash', 'ruined', 'destroyed', 'furious', 'angry'],
            'disappointment': ['disappointed', 'letdown', 'expected better', 'underwhelming',
                              'waste of time', 'boring', 'meh', 'overrated', 'overhyped'],
            'fear': ['scared', 'terrifying', 'creepy', 'disturbing', 'nightmare', 'horrifying',
                    'spine-chilling', 'jump scare', 'suspenseful'],
            'sadness': ['sad', 'depressing', 'heartbreaking', 'emotional', 'tear jerker', 'cried',
                       'touching', 'moving', 'tragic'],
            'surprise': ['shocked', 'unexpected', 'plot twist', 'surprised', 'didnt see coming',
                        'jaw dropped', 'mind blown', 'revelation'],
            'nostalgia': ['nostalgic', 'childhood', 'memories', 'throwback', 'classic', 'old school',
                         'brings back', 'remember when'],
            'confusion': ['confused', 'wtf', 'what the', 'makes no sense', 'plot holes',
                         'convoluted', 'complicated', 'lost me'],
            'anticipation': ['cant wait', 'excited for', 'looking forward', 'hoping', 'sequel',
                           'part 2', 'next movie', 'continuation'],
            'appreciation': ['cinematography', 'direction', 'acting', 'performance', 'technical',
                           'craft', 'artistry', 'visually stunning', 'well-made', 'thoughtful']
        }

        emotion_scores = {}
        words = text.split()

        for emotion, keywords in emotion_keywords.items():
            score = 0
            for keyword in keywords:
                if keyword in text:
                    score += text.count(keyword) * 0.2
                for word in words:
                    if keyword in word:
                        score += 0.1
            emotion_scores[emotion] = min(score, 1.0)

        return emotion_scores

class AdvancedMovieDetector:
    def __init__(self):
        print("Initializing enhanced movie detection system for Letterboxd...")
        # Load spaCy with only essential components
        try:
            self.nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
        except OSError:
            print("Warning: spaCy model not found. Using basic detection only.")
            self.nlp = None

        self.movie_titles = set()
        self.movie_patterns = []
        self.setup_detection_system()

    def setup_detection_system(self):
        """Setup comprehensive movie detection"""
        # Load all movies
        all_movies = COMPREHENSIVE_MOVIES.copy()

        # Try to load from Google Sheets if available
        try:
            additional_movies = self._load_from_google_sheets()
            if additional_movies:
                all_movies.extend(additional_movies)
        except:
            pass

        # Process all movie titles
        self._process_all_movies(all_movies)

        # Setup EntityRuler with aggressive patterns if spaCy is available
        if self.nlp:
            self._setup_entity_ruler()

        print(f"Detection system ready with {len(self.movie_titles)} movie patterns")

    def _load_from_google_sheets(self):
        """Try to load additional movies from Google Sheets"""
        try:
            import os
            if not os.path.exists(SERVICE_ACCOUNT_FILE):
                return None

            scopes = ['https://www.googleapis.com/auth/spreadsheets']
            creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scopes)
            client = gspread.authorize(creds)
            sheet = client.open_by_key(MOVIE_SHEET_ID).worksheet(MOVIE_WORKSHEET_NAME)
            all_data = sheet.get_all_records()
            df = pd.DataFrame(all_data)

            if NAME_COLUMN_IN_GSHEET in df.columns:
                movies = df[NAME_COLUMN_IN_GSHEET].dropna().astype(str).tolist()
                print(f"Loaded {len(movies)} additional movies from Google Sheets")
                return movies
        except Exception as e:
            print(f"Could not load from Google Sheets: {e}")
        return None

    def _process_all_movies(self, movies):
        """Process and clean all movie titles"""
        for movie in movies:
            if not movie or len(str(movie).strip()) < 2:
                continue

            movie = str(movie).strip()
            cleaned = self._clean_title(movie)
            if cleaned and self._is_valid_movie_title(cleaned):
                self.movie_titles.add(cleaned.lower())
                self._add_variations(cleaned)

        # Add abbreviations
        for abbr, full_name in MOVIE_ABBREVIATIONS.items():
            self.movie_titles.add(abbr.lower())
            self.movie_titles.add(full_name.lower())

    def _clean_title(self, title):
        """Clean movie title"""
        title = re.sub(r'\s*\(\d{4}\).*$', '', title)
        title = re.sub(r'\s+(movie|film)$', '', title, flags=re.IGNORECASE)
        title = re.sub(r'[^\w\s\'\-]', ' ', title)
        title = ' '.join(title.split())
        return title.strip()

    def _is_valid_movie_title(self, title):
        """Check if title is valid - very permissive now"""
        title_lower = title.lower().strip()
        if title_lower in STRICT_BLACKLIST:
            return False
        if len(title_lower) < 2:
            return False
        return True

    def _add_variations(self, title):
        """Add common variations"""
        title_lower = title.lower()
        if title_lower.startswith('the '):
            self.movie_titles.add(title_lower[4:])
        if "'" in title_lower:
            self.movie_titles.add(title_lower.replace("'", ""))
            self.movie_titles.add(title_lower.replace("'", ""))
        if ' ' in title_lower and len(title_lower.split()) == 2:
            self.movie_titles.add(title_lower.replace(' ', '-'))

    def _setup_entity_ruler(self):
        """Setup EntityRuler with comprehensive patterns"""
        if not self.nlp:
            return

        ruler = self.nlp.add_pipe("entity_ruler", before="ner")
        patterns = []
        for movie in self.movie_titles:
            patterns.append({"label": "MOVIE", "pattern": movie})
        ruler.add_patterns(patterns)
        print(f"Added {len(patterns)} patterns to EntityRuler")

    def detect_movies_comprehensive(self, text):
        """Comprehensive movie detection using multiple strategies"""
        if not text or not isinstance(text, str):
            return []

        text = text.lower().strip()
        found_movies = set()

        # Strategy 1: spaCy EntityRuler (if available)
        if self.nlp:
            try:
                doc = self.nlp(text)
                for ent in doc.ents:
                    if ent.label_ == "MOVIE":
                        movie = ent.text.strip()
                        if movie in MOVIE_ABBREVIATIONS:
                            movie = MOVIE_ABBREVIATIONS[movie]
                        found_movies.add(movie.title())
            except:
                pass

        # Strategy 2: Direct string matching with context
        found_movies.update(self._contextual_string_matching(text))

        # Strategy 3: Quoted text detection
        found_movies.update(self._detect_quoted_titles(text))

        # Strategy 4: Capitalized sequences
        found_movies.update(self._detect_capitalized_sequences(text))

        # Strategy 5: Letterboxd-specific patterns
        found_movies.update(self._detect_letterboxd_patterns(text))

        # Filter and validate results
        validated_movies = []
        for movie in found_movies:
            if self._validate_detection(movie, text):
                validated_movies.append(movie)

        return list(set(validated_movies))

    def _contextual_string_matching(self, text):
        """Find movies using context-aware string matching"""
        found = set()
        words = text.split()

        for i, word in enumerate(words):
            for length in range(1, min(6, len(words) - i + 1)):
                phrase = ' '.join(words[i:i+length])

                if phrase in self.movie_titles:
                    context_start = max(0, i - 5)
                    context_end = min(len(words), i + length + 5)
                    context = ' '.join(words[context_start:context_end])

                    if any(ctx_word in context for ctx_word in MOVIE_CONTEXT_WORDS):
                        if phrase in MOVIE_ABBREVIATIONS:
                            phrase = MOVIE_ABBREVIATIONS[phrase]
                        found.add(phrase.title())

        return found

    def _detect_quoted_titles(self, text):
        """Detect movie titles in quotes"""
        found = set()
        quote_patterns = [
            r'"([^"]{2,50})"',
            r"'([^']{2,50})'",
            r'\"([^\"]{2,50})\"'
        ]

        for pattern in quote_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if match.lower().strip() in self.movie_titles:
                    found.add(match.strip().title())

        return found

    def _detect_capitalized_sequences(self, text):
        """Detect capitalized word sequences that might be movie titles"""
        found = set()
        cap_pattern = r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b'
        matches = re.findall(cap_pattern, text)

        for match in matches:
            if match.lower() in self.movie_titles:
                found.add(match.title())

        return found

    def _detect_letterboxd_patterns(self, text):
        """Detect Letterboxd-specific movie mentions"""
        found = set()

        # Look for Letterboxd URL patterns
        letterboxd_url_pattern = r'letterboxd\.com/film/([^/\s]+)'
        matches = re.findall(letterboxd_url_pattern, text)
        for match in matches:
            movie_title = match.replace('-', ' ').title()
            if movie_title.lower() in self.movie_titles:
                found.add(movie_title)

        # Look for rating patterns
        rating_context_pattern = r'([^\n]{1,50})\s*[★☆]{3,5}'
        matches = re.findall(rating_context_pattern, text)
        for match in matches:
            words = match.strip().split()
            for i in range(len(words)):
                for length in range(1, min(5, len(words) - i + 1)):
                    phrase = ' '.join(words[i:i+length])
                    if phrase.lower() in self.movie_titles:
                        found.add(phrase.title())

        return found

    def _validate_detection(self, movie, full_text):
        """Validate movie detection - very permissive"""
        if not movie or len(movie.strip()) < 2:
            return False

        movie_lower = movie.lower().strip()

        if movie_lower in STRICT_BLACKLIST:
            return False
        if movie_lower in MOVIE_ABBREVIATIONS:
            return True
        if movie_lower in self.movie_titles:
            return True
        if len(movie.split()) > 1:
            return True

        return False

class RedditScraper:
    def __init__(self, reddit_config):
        try:
            self.reddit = praw.Reddit(**reddit_config)
        except Exception as e:
            print(f"Warning: Reddit connection failed: {e}")
            self.reddit = None
        self.emotion_analyzer = EmotionAnalyzer()

    def collect_data(self, subreddit_name, post_limit, comments_per_post, movie_detector, max_posts_to_scan):
        """Collect Reddit data with enhanced movie detection and emotion analysis"""
        all_data = []
        posts_with_movies = 0
        posts_scanned = 0
        skipped_posts = 0

        print(f"\nScraping {post_limit} posts WITH movies from r/{subreddit_name}...")
        print(f"Will scan up to {max_posts_to_scan} posts to find {post_limit} with detected movies")

        if not self.reddit:
            print("Reddit connection not available. Creating sample data instead...")
            return self._create_sample_data(post_limit)

        try:
            subreddit = self.reddit.subreddit(subreddit_name)
            post_sources = [
                ('hot', max_posts_to_scan // 3),
                ('new', max_posts_to_scan // 3),
                ('top', max_posts_to_scan // 3)
            ]

            for sort_method, limit in post_sources:
                if posts_with_movies >= post_limit:
                    break

                print(f"\nFetching {sort_method} posts...")

                if sort_method == 'hot':
                    posts = subreddit.hot(limit=limit)
                elif sort_method == 'new':
                    posts = subreddit.new(limit=limit)
                elif sort_method == 'top':
                    posts = subreddit.top(time_filter='week', limit=limit)

                for submission in posts:
                    posts_scanned += 1

                    if posts_with_movies >= post_limit:
                        break

                    post_text = f"{submission.title}. {submission.selftext}".strip()
                    found_movies = movie_detector.detect_movies_comprehensive(post_text)

                    comment_text = ""
                    comment_movies = []

                    if submission.num_comments > 0:
                        try:
                            submission.comments.replace_more(limit=0)
                            comments = [c.body for c in submission.comments[:comments_per_post]
                                      if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']]
                            comment_text = " ".join(comments[:3])

                            if comment_text:
                                comment_movies = movie_detector.detect_movies_comprehensive(comment_text)

                        except Exception as e:
                            print(f"Error processing comments for post {posts_scanned}: {e}")

                    all_found_movies = list(set(found_movies + comment_movies))

                    if not all_found_movies:
                        skipped_posts += 1
                        if posts_scanned % 50 == 0:
                            print(f"Scanned {posts_scanned} posts | Found {posts_with_movies}/{post_limit} with movies | Skipped {skipped_posts}")
                        continue

                    # We found movies! Process this post
                    posts_with_movies += 1

                    # Analyze emotions in the combined text
                    full_text_for_emotion = f"{post_text} {comment_text}"
                    emotion_results = self.emotion_analyzer.analyze_emotions(full_text_for_emotion)

                    # Store results
                    data_entry = {
                        'Title': submission.title,
                        'Author': str(submission.author) if submission.author else '[deleted]',
                        'All_Movies': all_found_movies,
                        'Primary_Movie': all_found_movies[0],
                        'Movie_Count': len(all_found_movies),
                        'Content': submission.selftext[:200] + '...' if len(submission.selftext) > 200 else submission.selftext,
                        'Comments_Sample': comment_text[:300] + '...' if len(comment_text) > 300 else comment_text,
                        'Timestamp_UTC': datetime.datetime.utcfromtimestamp(submission.created_utc),
                        'Sort_Method': sort_method
                    }

                    # Add emotion analysis results
                    data_entry.update(emotion_results)
                    all_data.append(data_entry)

                    if posts_with_movies % 25 == 0:
                        print(f"Progress: {posts_with_movies}/{post_limit} posts with movies | Scanned {posts_scanned} total | Skipped {skipped_posts}")

                    time.sleep(0.15)

        except Exception as e:
            print(f"Error during scraping: {e}")

        print(f"\nScraping complete!")
        print(f"Posts scanned: {posts_scanned}")
        print(f"Posts with movies found: {posts_with_movies}")
        print(f"Posts skipped (no movies): {skipped_posts}")
        if posts_scanned > 0:
            print(f"Success rate: {posts_with_movies/posts_scanned*100:.1f}%")

        return pd.DataFrame(all_data)

    def _create_sample_data(self, post_limit):
        """Create sample data when Reddit connection is not available"""
        print("Creating sample data for demonstration...")

        sample_movies = ["The Batman", "Dune", "Spider-Man No Way Home", "Everything Everywhere All at Once", "Top Gun Maverick"]
        sample_emotions = ["joy", "excitement", "love", "disappointment", "appreciation"]

        sample_data = []
        for i in range(min(post_limit, 50)):  # Create up to 50 sample records
            movie = sample_movies[i % len(sample_movies)]
            emotion = sample_emotions[i % len(sample_emotions)]

            data_entry = {
                'Title': f'Sample post about {movie} #{i+1}',
                'Author': f'user_{i+1}',
                'All_Movies': [movie],
                'Primary_Movie': movie,
                'Movie_Count': 1,
                'Content': f'This is a sample post discussing {movie}...',
                'Comments_Sample': f'Great movie! I loved {movie}...',
                'Timestamp_UTC': datetime.datetime.now() - datetime.timedelta(hours=i),
                'Sort_Method': 'sample',
                'emotion_1': emotion,
                'emotion_1_score': 0.8,
                'emotion_2': 'neutral',
                'emotion_2_score': 0.2,
                'emotion_3': 'neutral',
                'emotion_3_score': 0.0
            }
            sample_data.append(data_entry)

        return pd.DataFrame(sample_data)

def main():
    """Main execution function"""
    try:
        # Initialize enhanced movie detector
        print("=== Reddit Letterboxd Movie Sentiment Analysis - 1000 Posts ===")
        detector = AdvancedMovieDetector()

        # Initialize Reddit scraper
        scraper = RedditScraper(REDDIT_CONFIG)

        # Collect data - only posts with detected movies
        df = scraper.collect_data(SUBREDDIT_NAME, POST_LIMIT, COMMENTS_PER_POST, detector, MAX_POSTS_TO_SCAN)

        # Save results with fixed filename
        output_file = 'r_letterboxd_1000_posts.csv'
        df.to_csv(output_file, index=False)
        print(f"\nData saved to {output_file}")

        # Show comprehensive results
        print(f"\n=== DETECTION RESULTS ===")
        print(f"Total posts with movies collected: {len(df)}")

        if len(df) > 0:
            # Movie statistics
            all_movies = []
            for movies_list in df['All_Movies']:
                if isinstance(movies_list, list):
                    all_movies.extend(movies_list)
                elif isinstance(movies_list, str):
                    # Handle case where movies are stored as strings
                    all_movies.append(movies_list)

            if all_movies:
                movie_counts = Counter(all_movies)
                print(f"\nTop 20 detected movies:")
                for movie, count in movie_counts.most_common(20):
                    print(f"  {movie}: {count}")

            # Show distribution by sort method
            if 'Sort_Method' in df.columns:
                sort_distribution = df['Sort_Method'].value_counts()
                print(f"\nPosts by source:")
                for method, count in sort_distribution.items():
                    print(f"  {method}: {count}")

            # Show sample detections with emotions
            print(f"\nSample movie detections with emotions:")
            sample = df[['Title', 'Primary_Movie', 'Movie_Count', 'emotion_1', 'emotion_1_score']].head(15)
            for idx, row in sample.iterrows():
                title_short = row['Title'][:60] + '...' if len(row['Title']) > 60 else row['Title']
                print(f"  '{title_short}' -> {row['Primary_Movie']} | {row['emotion_1']} ({row['emotion_1_score']:.3f})")

            # Show emotion statistics
            print(f"\nTop emotions detected:")
            all_emotions = list(df['emotion_1']) + list(df['emotion_2']) + list(df['emotion_3'])
            emotion_counts = Counter(all_emotions)
            for emotion, count in emotion_counts.most_common(12):
                print(f"  {emotion}: {count}")

            # Show posts with multiple movies
            multi_movie_df = df[df['Movie_Count'] > 1]
            print(f"\nPosts with multiple movies detected: {len(multi_movie_df)}")
            print(f"Average movies per post: {df['Movie_Count'].mean():.2f}")
            print(f"Maximum movies in single post: {df['Movie_Count'].max()}")

            # Show time distribution
            if 'Timestamp_UTC' in df.columns:
                df['Hour'] = pd.to_datetime(df['Timestamp_UTC']).dt.hour
                hour_dist = df['Hour'].value_counts().head(5)
                print(f"\nTop posting hours (UTC):")
                for hour, count in hour_dist.items():
                    print(f"  {hour}:00 - {count} posts")

            # Quality metrics
            avg_content_length = df['Content'].str.len().mean()
            posts_with_comments = len(df[df['Comments_Sample'].str.len() > 10])

            print(f"\n=== QUALITY METRICS ===")
            print(f"Average content length: {avg_content_length:.0f} characters")
            print(f"Posts with meaningful comments: {posts_with_comments}")
            print(f"Unique movies detected: {len(set(all_movies))}")

        print(f"\n=== ANALYSIS COMPLETE ===")
        print(f"Dataset contains {len(df)} posts, all with detected movies")
        print(f"Data includes movie detection + top 3 emotions with scores for each post")
        print(f"Data saved as: r_letterboxd_1000_posts.csv")

        # Final status
        if len(df) >= 1000:
            print(f"\n=== SUCCESS! ===")
            print(f"✓ Successfully collected {len(df)} posts with movie mentions")
            print(f"✓ Ready for sentiment analysis and machine learning")
            print(f"✓ Comprehensive emotion data available")
            print(f"✓ File saved as: r_letterboxd_1000_posts.csv")
        else:
            print(f"\n=== PARTIAL SUCCESS ===")
            print(f"⚠ Collected {len(df)} posts (target was 1000)")
            print(f"⚠ Consider running again or adjusting detection parameters")
            print(f"✓ File saved as: r_letterboxd_1000_posts.csv")

    except Exception as e:
        print(f"Script failed: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

=== Reddit Letterboxd Movie Sentiment Analysis - 1000 Posts ===
Initializing enhanced movie detection system for Letterboxd...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Added 373 patterns to EntityRuler
Detection system ready with 373 movie patterns

Scraping 1000 posts WITH movies from r/letterboxd...
Will scan up to 3000 posts to find 1000 with detected movies

Fetching hot posts...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 25/1000 posts with movies | Scanned 49 total | Skipped 24


...

Progress: 950/1000 posts with movies | Scanned 1553 total | Skipped 603


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 975/1000 posts with movies | Scanned 1601 total | Skipped 626


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 1000/1000 posts with movies | Scanned 1648 total | Skipped 648

Scraping complete!
Posts scanned: 1649
Posts with movies found: 1000
Posts skipped (no movies): 648
Success rate: 60.6%

Data saved to r_letterboxd_1000_posts.csv

=== DETECTION RESULTS ===
Total posts with movies collected: 1000

Top 20 detected movies:
  Aquaman: 352
  Up: 211
  Captain America: 129
  Saw: 124
  Favourite: 64
  Batman: 29
  Star Wars: 28
  Alien: 24
  Indiana Jones: 23
  Jurassic Park: 23
  Lord Of The Rings: 23
  Dune: 20
  Scream: 20
  2001 A Space Odyssey: 20
  Iron Man: 20
  Blade Runner: 19
  Parasite: 17
  Spider-Man: 17
  Portrait Of A Lady On Fire: 17
  Raiders Of The Lost Ark: 16

Posts by source:
  hot: 595
  new: 405

Sample movie detections with emotions:
  'August Profile Swap Megathread!' -> Up | love (0.980)
  'Celine Song shares her thoughts about one of the top letterb...' -> Saw | neutral (0.736)
  'Will Smith Loses Bad Boys Director Michael Bay on Upcoming M...' -> Up | neutr

### Visualizations

In [ ]:
import pandas as pd
from collections import Counter
import ast

# Load the CSV
df = pd.read_csv("r_letterboxd_1000_posts.csv")

# Convert each "All_Movies" entry to a list
def convert_to_list(cell):
    if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == '[]':
        return []
    try:
        # Use ast.literal_eval to safely parse the string representation of a list
        return [movie.strip().lower() for movie in ast.literal_eval(cell)]
    except (ValueError, SyntaxError):
        print(f"Warning: Could not parse '{cell}'. Skipping this row.")
        return []

df["All_Movies"] = df["All_Movies"].apply(convert_to_list)

# Count and sort movie mentions
movie_counter = Counter()
for movie_list in df["All_Movies"]:
    movie_counter.update(movie_list)

# Sort by frequency (descending)
movie_counts = dict(sorted(movie_counter.items(), key=lambda item: item[1], reverse=True))

# Print each movie and its count
for movie, count in movie_counts.items():
    print(f"{movie}: {count}")

aquaman: 352
up: 211
captain america: 129
saw: 124
favourite: 64
batman: 29
star wars: 28
alien: 24
indiana jones: 23
jurassic park: 23
lord of the rings: 23
dune: 20
scream: 20
2001 a space odyssey: 20
iron man: 20
blade runner: 19
parasite: 17
spider-man: 17
portrait of a lady on fire: 17
raiders of the lost ark: 16
shrek: 14
jaws: 14
superman: 14
mad max: 13
harry potter: 13
psycho: 13
rocky: 12
empire strikes back: 11
solo: 11
phantom thread: 11
john wick: 11
terminator: 10
mulholland drive: 9
big lebowski: 9
halloween: 9
goodfellas: 9
schindlers list: 8
wolf of wall street: 8
godfather: 8
casablanca: 8
the matrix: 8
toy story: 8
revenge of the sith: 8
marvel cinematic universe: 8
get out: 8
citizen kane: 8
pulp fiction: 7
persona: 7
the incredibles: 7
james bond: 7
the shining: 7
how to train your dragon: 7
the godfather: 7
burning: 6
die hard: 6
predator: 6
finding nemo: 6
back to the future: 6
the phantom menace: 6
lady bird: 6
mission impossible: 6
superbad: 5
casino: 5
raging 

**Top 10 posts r/Letterboxd**

In [ ]:
import matplotlib.pyplot as plt

# Choose how many top movies to show
TOP_N = 10

# Get top N items
top_movies = list(movie_counts.items())[:TOP_N]
movie_names, counts = zip(*top_movies)

# Plotting
plt.figure(figsize=(12, 6))
plt.barh(movie_names[::-1], counts[::-1], color='lavender')  # reverse for descending top-down
plt.xlabel("Mention Count")
plt.title(f"Top {TOP_N} Most Mentioned Movies in r/letterboxd")
plt.tight_layout()
plt.show()

**Average Sentiment Scores per Day**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Load the data
df = pd.read_csv("r_letterboxd_1000_posts.csv")

# Extract date from 'Timestamp_UTC'
df['Date'] = pd.to_datetime(df['Timestamp_UTC'], errors='coerce').dt.date

# Parse emotions from individual columns (emotion_1, emotion_2, emotion_3 with their scores)
def parse_emotions_from_columns(row):
    emotions = {}
    # Get emotion scores from the separate columns
    emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']
    score_columns = ['emotion_1_score', 'emotion_2_score', 'emotion_3_score']

    for i, (emo_col, score_col) in enumerate(zip(emotion_columns, score_columns)):
        if emo_col in row and score_col in row:
            if pd.notna(row[emo_col]) and pd.notna(row[score_col]):
                emotion = str(row[emo_col]).lower()
                score = float(row[score_col])
                emotions[emotion] = score
    return emotions

# Apply parsing and expand to new columns
emotion_df = df.apply(parse_emotions_from_columns, axis=1).apply(pd.Series)
emotion_df['Date'] = df['Date']

# Group by date and average
daily_sentiments = emotion_df.groupby('Date').mean().reset_index()

# Melt for seaborn
melted = daily_sentiments.melt(id_vars='Date', var_name='Emotion', value_name='Score')

# Plot
plt.figure(figsize=(16, 6))
sns.lineplot(data=melted, x='Date', y='Score', hue='Emotion', marker='o')
plt.title("Average Sentiment Scores per Day in r/letterboxd", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Average Sentiment Score", fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Sentiment", fontsize=10, title_fontsize=12)
plt.tight_layout()
plt.show()

# **Final Code r/Truefilm**

### CODE

In [ ]:
import praw
import pandas as pd
import time
import gspread
from google.oauth2.service_account import Credentials
import spacy
import datetime
import re
from collections import Counter
import requests
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download required NLTK data
try:
    nltk.download('vader_lexicon', quiet=True)
except:
    pass

# --- Configuration ---
SERVICE_ACCOUNT_FILE = 'YOUR_SERVICE_ACCOUNT_FILE'
MOVIE_SHEET_ID = 'YOUR_MOVIE_SHEET_ID'
MOVIE_WORKSHEET_NAME = 'movies'
NAME_COLUMN_IN_GSHEET = 'Movie Names'

SUBREDDIT_NAME = 'TrueFilm'  # Changed from 'letterboxd' to 'TrueFilm'
POST_LIMIT = 1000  # Increased from 100 to 1000
COMMENTS_PER_POST = 3
OUTPUT_CSV = 'r_truefilm_1000_posts.csv'  # Fixed filename - always saves as this name
MAX_POSTS_TO_SCAN = 3000  # Increased from 500 to 3000 to find 1000 posts with movies

# Reddit API Configuration
REDDIT_CONFIG = {
    'client_id': 'YOUR_CLIENT_ID',
    'client_secret': "YOUR_CLIENT_SECRET",
    'user_agent': 'TrueFilmSentimentBot by /u/anonymous'  # Updated user agent
}

# Expanded movie database with popular movies, franchises, and series
COMPREHENSIVE_MOVIES = [
    # Marvel Cinematic Universe
    "Iron Man", "Captain America", "Thor", "The Avengers", "Hulk", "Black Widow",
    "Guardians of the Galaxy", "Ant-Man", "Doctor Strange", "Spider-Man", "Black Panther",
    "Captain Marvel", "Avengers Endgame", "Infinity War", "Civil War", "Winter Soldier",
    "Age of Ultron", "Ragnarok", "Homecoming", "Far From Home", "No Way Home",
    "Eternals", "Shang-Chi", "Multiverse of Madness", "Love and Thunder", "Wakanda Forever",

    # DC Universe
    "Batman", "Superman", "Wonder Woman", "Justice League", "Aquaman", "The Flash",
    "Green Lantern", "Suicide Squad", "Birds of Prey", "Joker", "The Batman",
    "Man of Steel", "Batman v Superman", "Dark Knight", "Dark Knight Rises", "Batman Begins",

    # Star Wars
    "Star Wars", "Empire Strikes Back", "Return of the Jedi", "Phantom Menace",
    "Attack of the Clones", "Revenge of the Sith", "Force Awakens", "Last Jedi",
    "Rise of Skywalker", "Rogue One", "Solo", "Mandalorian",

    # Classic/Popular Films
    "The Shawshank Redemption", "The Godfather", "Pulp Fiction", "Forrest Gump",
    "Inception", "The Matrix", "Goodfellas", "Casablanca", "Citizen Kane",
    "Schindler's List", "Lord of the Rings", "Fellowship of the Ring", "Two Towers",
    "Return of the King", "The Hobbit", "Titanic", "Avatar", "Jurassic Park",
    "E.T.", "Jaws", "Back to the Future", "Indiana Jones", "Raiders of the Lost Ark",

    # Horror
    "Halloween", "Friday the 13th", "Nightmare on Elm Street", "The Exorcist",
    "The Shining", "Psycho", "Scream", "Get Out", "Hereditary", "Midsommar",
    "It", "The Conjuring", "Insidious", "Paranormal Activity", "Saw", "Texas Chainsaw Massacre",

    # Action/Thriller
    "John Wick", "Fast and Furious", "Mission Impossible", "Die Hard", "Lethal Weapon",
    "Terminator", "Alien", "Predator", "Rocky", "Rambo", "Mad Max", "Blade Runner",
    "The Bourne Identity", "James Bond", "Casino Royale", "Skyfall", "Spectre",

    # Comedy
    "Anchorman", "Superbad", "Pineapple Express", "Step Brothers", "Talladega Nights",
    "Zoolander", "Meet the Parents", "There's Something About Mary", "Dumb and Dumber",
    "The Hangover", "Bridesmaids", "Ghostbusters", "Groundhog Day", "Big Lebowski",

    # Drama
    "The Departed", "Scarface", "Casino", "Taxi Driver", "Raging Bull",
    "The Wolf of Wall Street", "Django Unchained", "Kill Bill", "Inglourious Basterds",
    "Once Upon a Time in Hollywood", "Parasite", "Nomadland", "Minari", "Sound of Metal",

    # Animated
    "Toy Story", "Finding Nemo", "The Incredibles", "Up", "Wall-E", "Inside Out",
    "Coco", "Moana", "Frozen", "The Lion King", "Beauty and the Beast", "Aladdin",
    "Shrek", "How to Train Your Dragon", "Despicable Me", "Minions", "Ice Age",

    # Recent Popular
    "Top Gun Maverick", "Everything Everywhere All at Once", "The Batman", "Doctor Strange",
    "Thor Love and Thunder", "Black Panther Wakanda Forever", "Avatar Way of Water",
    "Glass Onion", "Knives Out", "Don't Look Up", "Dune", "No Time to Die", "Spider-Man No Way Home",

    # TrueFilm Popular/Art House Films (added for TrueFilm context)
    "Mulholland Drive", "2001 A Space Odyssey", "Vertigo", "8½", "Persona", "Tokyo Story",
    "The Rules of the Game", "Bicycle Thieves", "Singin' in the Rain", "Some Like It Hot",
    "North by Northwest", "Psycho", "La Dolce Vita", "The 400 Blows", "Breathless",
    "Contempt", "Pierrot le Fou", "Cries and Whispers", "Scenes from a Marriage",
    "Annie Hall", "Manhattan", "Taxi Driver", "Raging Bull", "GoodFellas",
    "The Seventh Seal", "Wild Strawberries", "Fanny and Alexander", "Amour",
    "Blue Is the Warmest Color", "Call Me by Your Name", "Moonlight", "Lady Bird",
    "Portrait of a Lady on Fire", "The Handmaiden", "Burning", "Shoplifters",
    "Roma", "The Favourite", "Marriage Story", "Phantom Thread", "There Will Be Blood"
]

# Movie abbreviations and slang (updated for TrueFilm context)
MOVIE_ABBREVIATIONS = {
    'mcu': 'marvel cinematic universe',
    'dceu': 'dc extended universe',
    'lotr': 'lord of the rings',
    'sw': 'star wars',
    'hp': 'harry potter',
    'gotg': 'guardians of the galaxy',
    'potc': 'pirates of the caribbean',
    'tdk': 'the dark knight',
    'tdkr': 'the dark knight rises',
    'bb': 'batman begins',
    'bvs': 'batman v superman',
    'mos': 'man of steel',
    'jl': 'justice league',
    'ss': 'suicide squad',
    'ww': 'wonder woman',
    'am': 'aquaman',
    'sm': 'spider-man',
    'im': 'iron man',
    'ca': 'captain america',
    'tws': 'the winter soldier',
    'cw': 'civil war',
    'aou': 'age of ultron',
    'tfa': 'the force awakens',
    'tlj': 'the last jedi',
    'tros': 'the rise of skywalker',
    'rotj': 'return of the jedi',
    'esb': 'empire strikes back',
    'anh': 'a new hope',
    'rots': 'revenge of the sith',
    'aotc': 'attack of the clones',
    'tpm': 'the phantom menace',
    'iw': 'infinity war',
    'eg': 'endgame',
    'nwh': 'no way home',
    'ffh': 'far from home',
    'hc': 'homecoming',
    'ragnarok': 'thor ragnarok',
    'bp': 'black panther',
    'cm': 'captain marvel',
    'ds': 'doctor strange',
    'gotg2': 'guardians of the galaxy vol 2',
    'antman': 'ant-man',
    'jp': 'jurassic park',
    'bttf': 'back to the future',
    'rotla': 'raiders of the lost ark',
    'mi': 'mission impossible',
    # TrueFilm specific abbreviations
    'potlaofl': 'portrait of a lady on fire',
    'eeoao': 'everything everywhere all at once',
    'twbb': 'there will be blood',
    'md': 'mulholland drive',
    '2001': '2001 a space odyssey'
}

# Minimal blacklist - only truly generic words
STRICT_BLACKLIST = [
    'movie', 'movies', 'film', 'films', 'cinema', 'flick', 'picture', 'show',
    'the', 'a', 'an', 'and', 'or', 'in', 'on', 'at', 'to', 'for', 'of', 'with',
    'this', 'that', 'my', 'your', 'his', 'her', 'our', 'their', 'it', 'its',
    'truefilm', 'discussion', 'analysis', 'critique', 'theory'  # Added TrueFilm specific terms
]

# Movie context indicators - words that suggest movie discussion (enhanced for TrueFilm)
MOVIE_CONTEXT_WORDS = [
    'watch', 'watched', 'watching', 'see', 'seen', 'saw', 'love', 'loved', 'hate', 'hated',
    'like', 'liked', 'dislike', 'enjoy', 'enjoyed', 'favorite', 'favourite', 'fav',
    'best', 'worst', 'good', 'bad', 'great', 'amazing', 'terrible', 'awful',
    'recommend', 'recommended', 'suggest', 'review', 'reviewed', 'rating', 'rated',
    'director', 'directed', 'starring', 'stars', 'cast', 'actor', 'actress',
    'sequel', 'prequel', 'remake', 'reboot', 'trailer', 'teaser', 'clip',
    'spoiler', 'spoilers', 'plot', 'story', 'character', 'characters',
    'scene', 'scenes', 'ending', 'beginning', 'climax', 'twist',
    # TrueFilm specific context words
    'analysis', 'critique', 'discussion', 'theory', 'interpretation', 'symbolism',
    'auteur', 'cinephile', 'filmmaking', 'cinematography', 'mise-en-scene',
    'screenplay', 'score', 'soundtrack', 'editing', 'production', 'festival',
    'arthouse', 'art-house', 'foreign', 'subtitled', 'masterpiece', 'classic',
    'underrated', 'overrated', 'thematic', 'narrative', 'visual', 'aesthetic',
    'philosophical', 'psychological', 'sociological', 'cultural', 'historical'
]

class EmotionAnalyzer:
    def __init__(self):
        """Initialize emotion analysis tools"""
        try:
            self.sia = SentimentIntensityAnalyzer()
        except:
            print("Warning: VADER sentiment analyzer not available")
            self.sia = None

    def analyze_emotions(self, text):
        """Analyze emotions in text and return top 3 emotions with scores"""
        if not text or not isinstance(text, str):
            return {
                'emotion_1': 'neutral',
                'emotion_1_score': 0.0,
                'emotion_2': 'neutral',
                'emotion_2_score': 0.0,
                'emotion_3': 'neutral',
                'emotion_3_score': 0.0
            }

        # Clean text for analysis
        text = text.lower().strip()

        # Get basic sentiment using TextBlob
        try:
            blob = TextBlob(text)
            polarity = blob.sentiment.polarity  # -1 to 1
            subjectivity = blob.sentiment.subjectivity  # 0 to 1
        except:
            polarity = 0.0
            subjectivity = 0.0

        # Get VADER sentiment scores
        if self.sia:
            try:
                vader_scores = self.sia.polarity_scores(text)
                compound = vader_scores['compound']
                pos_score = vader_scores['pos']
                neg_score = vader_scores['neg']
                neu_score = vader_scores['neu']
            except:
                compound = pos_score = neg_score = neu_score = 0.0
        else:
            compound = pos_score = neg_score = neu_score = 0.0

        # Emotion keyword detection with scores
        emotions = self._detect_emotion_keywords(text)

        # Combine different approaches to determine top emotions
        emotion_scores = {
            'joy': max(pos_score, polarity if polarity > 0 else 0) + emotions.get('joy', 0),
            'excitement': emotions.get('excitement', 0) + (pos_score * 0.8),
            'love': emotions.get('love', 0) + (pos_score * 0.6),
            'anger': neg_score + emotions.get('anger', 0),
            'disappointment': emotions.get('disappointment', 0) + (neg_score * 0.7),
            'fear': emotions.get('fear', 0) + (neg_score * 0.5),
            'sadness': emotions.get('sadness', 0) + (neg_score * 0.6),
            'surprise': emotions.get('surprise', 0) + (abs(polarity) * subjectivity),
            'nostalgia': emotions.get('nostalgia', 0),
            'confusion': emotions.get('confusion', 0) + (subjectivity * 0.5),
            'anticipation': emotions.get('anticipation', 0),
            'admiration': emotions.get('admiration', 0),  # Added for TrueFilm context
            'intellectual': emotions.get('intellectual', 0),  # Added for analytical discussions
            'neutral': neu_score
        }

        # Get top 3 emotions
        sorted_emotions = sorted(emotion_scores.items(), key=lambda x: x[1], reverse=True)
        top_3 = sorted_emotions[:3]

        # Ensure we have 3 emotions
        while len(top_3) < 3:
            top_3.append(('neutral', 0.0))

        return {
            'emotion_1': top_3[0][0],
            'emotion_1_score': round(top_3[0][1], 3),
            'emotion_2': top_3[1][0],
            'emotion_2_score': round(top_3[1][1], 3),
            'emotion_3': top_3[2][0],
            'emotion_3_score': round(top_3[2][1], 3)
        }

    def _detect_emotion_keywords(self, text):
        """Detect emotion-specific keywords and return scores (enhanced for TrueFilm)"""
        emotion_keywords = {
            'joy': ['amazing', 'awesome', 'fantastic', 'incredible', 'brilliant', 'outstanding',
                   'excellent', 'wonderful', 'perfect', 'beautiful', 'hilarious', 'funny'],
            'excitement': ['excited', 'thrilled', 'pumped', 'hyped', 'anticipating', 'cant wait',
                          'looking forward', 'epic', 'incredible', 'mind-blowing'],
            'love': ['love', 'adore', 'favorite', 'favourite', 'obsessed', 'amazing chemistry',
                    'perfect cast', 'masterpiece', 'classic'],
            'anger': ['hate', 'terrible', 'awful', 'worst', 'horrible', 'disgusting', 'garbage',
                     'trash', 'ruined', 'destroyed', 'furious', 'angry'],
            'disappointment': ['disappointed', 'letdown', 'expected better', 'underwhelming',
                              'waste of time', 'boring', 'meh', 'overrated', 'overhyped'],
            'fear': ['scared', 'terrifying', 'creepy', 'disturbing', 'nightmare', 'horrifying',
                    'spine-chilling', 'jump scare', 'suspenseful'],
            'sadness': ['sad', 'depressing', 'heartbreaking', 'emotional', 'tear jerker', 'cried',
                       'touching', 'moving', 'tragic'],
            'surprise': ['shocked', 'unexpected', 'plot twist', 'surprised', 'didnt see coming',
                        'jaw dropped', 'mind blown', 'revelation'],
            'nostalgia': ['nostalgic', 'childhood', 'memories', 'throwback', 'classic', 'old school',
                         'brings back', 'remember when'],
            'confusion': ['confused', 'wtf', 'what the', 'makes no sense', 'plot holes',
                         'convoluted', 'complicated', 'lost me'],
            'anticipation': ['cant wait', 'excited for', 'looking forward', 'hoping', 'sequel',
                           'part 2', 'next movie', 'continuation'],
            'admiration': ['admire', 'respect', 'appreciate', 'genius', 'talented', 'skillful',
                          'craftsmanship', 'artistry', 'visionary', 'innovative'],  # Added for film admiration
            'intellectual': ['analysis', 'theory', 'interpretation', 'symbolism', 'metaphor',
                           'thematic', 'philosophical', 'thought-provoking', 'complex', 'nuanced']  # Added for analytical discussions
        }

        emotion_scores = {}
        words = text.split()

        for emotion, keywords in emotion_keywords.items():
            score = 0
            for keyword in keywords:
                # Count occurrences of keywords
                if keyword in text:
                    score += text.count(keyword) * 0.2

                # Check for partial matches in words
                for word in words:
                    if keyword in word:
                        score += 0.1

            emotion_scores[emotion] = min(score, 1.0)  # Cap at 1.0

        return emotion_scores

class AdvancedMovieDetector:
    def __init__(self):
        print("Initializing enhanced movie detection system for TrueFilm...")
        # Load spaCy with only essential components
        self.nlp = spacy.load("en_core_web_sm", disable=["parser", "lemmatizer"])
        self.movie_titles = set()
        self.movie_patterns = []
        self.setup_detection_system()

    def setup_detection_system(self):
        """Setup comprehensive movie detection"""
        # Load all movies
        all_movies = COMPREHENSIVE_MOVIES.copy()

        # Try to load from Google Sheets if available
        try:
            additional_movies = self._load_from_google_sheets()
            if additional_movies:
                all_movies.extend(additional_movies)
        except:
            pass

        # Process all movie titles
        self._process_all_movies(all_movies)

        # Setup EntityRuler with aggressive patterns
        self._setup_entity_ruler()

        print(f"Detection system ready with {len(self.movie_titles)} movie patterns")

    def _load_from_google_sheets(self):
        """Try to load additional movies from Google Sheets"""
        try:
            import os
            if not os.path.exists(SERVICE_ACCOUNT_FILE):
                return None

            scopes = ['https://www.googleapis.com/auth/spreadsheets']
            creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_FILE, scopes=scopes)
            client = gspread.authorize(creds)
            sheet = client.open_by_key(MOVIE_SHEET_ID).worksheet(MOVIE_WORKSHEET_NAME)
            all_data = sheet.get_all_records()
            df = pd.DataFrame(all_data)

            if NAME_COLUMN_IN_GSHEET in df.columns:
                movies = df[NAME_COLUMN_IN_GSHEET].dropna().astype(str).tolist()
                print(f"Loaded {len(movies)} additional movies from Google Sheets")
                return movies
        except Exception as e:
            print(f"Could not load from Google Sheets: {e}")
        return None

    def _process_all_movies(self, movies):
        """Process and clean all movie titles"""
        for movie in movies:
            if not movie or len(str(movie).strip()) < 2:
                continue

            movie = str(movie).strip()

            # Add original title
            cleaned = self._clean_title(movie)
            if cleaned and self._is_valid_movie_title(cleaned):
                self.movie_titles.add(cleaned.lower())

                # Add variations
                self._add_variations(cleaned)

        # Add abbreviations
        for abbr, full_name in MOVIE_ABBREVIATIONS.items():
            self.movie_titles.add(abbr.lower())
            self.movie_titles.add(full_name.lower())

    def _clean_title(self, title):
        """Clean movie title"""
        # Remove year in parentheses
        title = re.sub(r'\s*\(\d{4}\).*$', '', title)

        # Remove common suffixes
        title = re.sub(r'\s+(movie|film)$', '', title, flags=re.IGNORECASE)

        # Clean special characters but keep apostrophes and hyphens
        title = re.sub(r'[^\w\s\'\-]', ' ', title)

        # Normalize whitespace
        title = ' '.join(title.split())

        return title.strip()

    def _is_valid_movie_title(self, title):
        """Check if title is valid - very permissive now"""
        title_lower = title.lower().strip()

        # Only reject if it's in the strict blacklist
        if title_lower in STRICT_BLACKLIST:
            return False

        # Must be at least 2 characters
        if len(title_lower) < 2:
            return False

        return True

    def _add_variations(self, title):
        """Add common variations"""
        title_lower = title.lower()

        # Add version without "the"
        if title_lower.startswith('the '):
            self.movie_titles.add(title_lower[4:])

        # Add version without apostrophes
        if "'" in title_lower:
            self.movie_titles.add(title_lower.replace("'", ""))
            self.movie_titles.add(title_lower.replace("'", ""))

        # Add hyphenated version
        if ' ' in title_lower and len(title_lower.split()) == 2:
            self.movie_titles.add(title_lower.replace(' ', '-'))

    def _setup_entity_ruler(self):
        """Setup EntityRuler with comprehensive patterns"""
        ruler = self.nlp.add_pipe("entity_ruler", before="ner")

        patterns = []
        for movie in self.movie_titles:
            patterns.append({"label": "MOVIE", "pattern": movie})

        ruler.add_patterns(patterns)
        print(f"Added {len(patterns)} patterns to EntityRuler")

    def detect_movies_comprehensive(self, text):
        """Comprehensive movie detection using multiple strategies"""
        if not text or not isinstance(text, str):
            return []

        text = text.lower().strip()
        found_movies = set()

        # Strategy 1: spaCy EntityRuler
        doc = self.nlp(text)
        for ent in doc.ents:
            if ent.label_ == "MOVIE":
                movie = ent.text.strip()
                if movie in MOVIE_ABBREVIATIONS:
                    movie = MOVIE_ABBREVIATIONS[movie]
                found_movies.add(movie.title())

        # Strategy 2: Direct string matching with context
        found_movies.update(self._contextual_string_matching(text))

        # Strategy 3: Quoted text detection
        found_movies.update(self._detect_quoted_titles(text))

        # Strategy 4: Capitalized sequences
        found_movies.update(self._detect_capitalized_sequences(text))

        # Strategy 5: TrueFilm-specific patterns (analysis context)
        found_movies.update(self._detect_truefilm_patterns(text))

        # Filter and validate results
        validated_movies = []
        for movie in found_movies:
            if self._validate_detection(movie, text):
                validated_movies.append(movie)

        return list(set(validated_movies))  # Remove duplicates

    def _contextual_string_matching(self, text):
        """Find movies using context-aware string matching"""
        found = set()
        words = text.split()

        for i, word in enumerate(words):
            # Check if current word or phrase matches a movie
            for length in range(1, min(6, len(words) - i + 1)):  # Check up to 5-word phrases
                phrase = ' '.join(words[i:i+length])

                if phrase in self.movie_titles:
                    # Check for movie context nearby
                    context_start = max(0, i - 5)
                    context_end = min(len(words), i + length + 5)
                    context = ' '.join(words[context_start:context_end])

                    if any(ctx_word in context for ctx_word in MOVIE_CONTEXT_WORDS):
                        # Expand abbreviations
                        if phrase in MOVIE_ABBREVIATIONS:
                            phrase = MOVIE_ABBREVIATIONS[phrase]
                        found.add(phrase.title())

        return found

    def _detect_quoted_titles(self, text):
        """Detect movie titles in quotes"""
        found = set()

        # Find text in quotes
        quote_patterns = [
            r'"([^"]{2,50})"',  # Double quotes
            r"'([^']{2,50})'",  # Single quotes
            r'\"([^\"]{2,50})\"'  # Escaped quotes
        ]

        for pattern in quote_patterns:
            matches = re.findall(pattern, text)
            for match in matches:
                if match.lower().strip() in self.movie_titles:
                    found.add(match.strip().title())

        return found

    def _detect_capitalized_sequences(self, text):
        """Detect capitalized word sequences that might be movie titles"""
        found = set()

        # Find sequences of capitalized words
        cap_pattern = r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b'
        matches = re.findall(cap_pattern, text)

        for match in matches:
            if match.lower() in self.movie_titles:
                found.add(match.title())

        return found

    def _detect_truefilm_patterns(self, text):
        """Detect TrueFilm-specific movie mentions (analytical context)"""
        found = set()

        # Look for analysis patterns with movie titles
        analysis_patterns = [
            r'analysis of ([^.,;]{2,40})',
            r'discussion about ([^.,;]{2,40})',
            r'thoughts on ([^.,;]{2,40})',
            r'critique of ([^.,;]{2,40})',
            r'interpretation of ([^.,;]{2,40})',
            r'review of ([^.,;]{2,40})'
        ]

        for pattern in analysis_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE)
            for match in matches:
                cleaned_match = match.strip().lower()
                if cleaned_match in self.movie_titles:
                    found.add(match.strip().title())

        # Look for director mentions with films
        director_pattern = r"([a-z\s']+)'s ([^.,;]{2,40})"
        matches = re.findall(director_pattern, text, re.IGNORECASE)
        for director, film in matches:
            if film.strip().lower() in self.movie_titles:
                found.add(film.strip().title())

        return found

    def _validate_detection(self, movie, full_text):
        """Validate movie detection - very permissive"""
        if not movie or len(movie.strip()) < 2:
            return False

        movie_lower = movie.lower().strip()

        # Reject only strict blacklist items
        if movie_lower in STRICT_BLACKLIST:
            return False

        # Accept if it's a known abbreviation
        if movie_lower in MOVIE_ABBREVIATIONS:
            return True

        # Accept if it's in our movie database
        if movie_lower in self.movie_titles:
            return True

        # Accept multi-word titles (more likely to be specific)
        if len(movie.split()) > 1:
            return True

        return False

class RedditScraper:
    def __init__(self, reddit_config):
        self.reddit = praw.Reddit(**reddit_config)
        self.emotion_analyzer = EmotionAnalyzer()

    def collect_data(self, subreddit_name, post_limit, comments_per_post, movie_detector, max_posts_to_scan):
        """Collect Reddit data with enhanced movie detection and emotion analysis - skip posts without movies"""
        all_data = []
        posts_with_movies = 0
        posts_scanned = 0
        skipped_posts = 0

        print(f"\nScraping {post_limit} posts WITH movies from r/{subreddit_name}...")
        print(f"Will scan up to {max_posts_to_scan} posts to find {post_limit} with detected movies")

        try:
            subreddit = self.reddit.subreddit(subreddit_name)

            # Mix different sorting methods to get more diverse posts
            post_sources = [
                ('hot', max_posts_to_scan // 3),
                ('new', max_posts_to_scan // 3),
                ('top', max_posts_to_scan // 3)
            ]

            for sort_method, limit in post_sources:
                if posts_with_movies >= post_limit:
                    break

                print(f"\nFetching {sort_method} posts...")

                if sort_method == 'hot':
                    posts = subreddit.hot(limit=limit)
                elif sort_method == 'new':
                    posts = subreddit.new(limit=limit)
                elif sort_method == 'top':
                    posts = subreddit.top(time_filter='week', limit=limit)

                for submission in posts:
                    posts_scanned += 1

                    # Check if we've found enough posts with movies
                    if posts_with_movies >= post_limit:
                        break

                    # Combine title and content for analysis
                    post_text = f"{submission.title}. {submission.selftext}".strip()

                    # Detect movies in post
                    found_movies = movie_detector.detect_movies_comprehensive(post_text)

                    # Also check comments for additional movies
                    comment_text = ""
                    comment_movies = []

                    if submission.num_comments > 0:
                        try:
                            submission.comments.replace_more(limit=0)
                            comments = [c.body for c in submission.comments[:comments_per_post]
                                      if hasattr(c, 'body') and c.body not in ['[deleted]', '[removed]']]
                            comment_text = " ".join(comments[:3])  # Limit comment text

                            if comment_text:
                                comment_movies = movie_detector.detect_movies_comprehensive(comment_text)

                        except Exception as e:
                            print(f"Error processing comments for post {posts_scanned}: {e}")

                    # Combine all found movies
                    all_found_movies = list(set(found_movies + comment_movies))

                    # Skip this post if no movies were detected
                    if not all_found_movies:
                        skipped_posts += 1
                        if posts_scanned % 50 == 0:
                            print(f"Scanned {posts_scanned} posts | Found {posts_with_movies}/{post_limit} with movies | Skipped {skipped_posts}")
                        continue

                    # We found movies! Process this post
                    posts_with_movies += 1

                    # Analyze emotions in the combined text
                    full_text_for_emotion = f"{post_text} {comment_text}"
                    emotion_results = self.emotion_analyzer.analyze_emotions(full_text_for_emotion)

                    # Store results (no 'Unknown' movies anymore)
                    data_entry = {
                        'Title': submission.title,
                        'Author': str(submission.author) if submission.author else '[deleted]',
                        'All_Movies': all_found_movies,
                        'Primary_Movie': all_found_movies[0],  # Always has at least one movie
                        'Movie_Count': len(all_found_movies),
                        'Content': submission.selftext[:200] + '...' if len(submission.selftext) > 200 else submission.selftext,
                        'Comments_Sample': comment_text[:300] + '...' if len(comment_text) > 300 else comment_text,
                        'Timestamp_UTC': datetime.datetime.utcfromtimestamp(submission.created_utc),
                        'Sort_Method': sort_method
                    }

                    # Add emotion analysis results
                    data_entry.update(emotion_results)
                    all_data.append(data_entry)

                    if posts_with_movies % 25 == 0:
                        print(f"Progress: {posts_with_movies}/{post_limit} posts with movies | Scanned {posts_scanned} total | Skipped {skipped_posts}")

                    time.sleep(0.15)  # Slightly reduced rate limiting for efficiency

        except Exception as e:
            print(f"Error during scraping: {e}")

        print(f"\nScraping complete!")
        print(f"Posts scanned: {posts_scanned}")
        print(f"Posts with movies found: {posts_with_movies}")
        print(f"Posts skipped (no movies): {skipped_posts}")
        print(f"Success rate: {posts_with_movies/posts_scanned*100:.1f}%")

        return pd.DataFrame(all_data)

def main():
    """Main execution function"""
    try:
        # Initialize enhanced movie detector
        print("=== Reddit TrueFilm Movie Sentiment Analysis - 1000 Posts ===")
        detector = AdvancedMovieDetector()

        # Initialize Reddit scraper
        scraper = RedditScraper(REDDIT_CONFIG)

        # Collect data - only posts with detected movies
        df = scraper.collect_data(SUBREDDIT_NAME, POST_LIMIT, COMMENTS_PER_POST, detector, MAX_POSTS_TO_SCAN)

        # Save results - ALWAYS save as 'r_truefilm_1000_posts.csv'
        df.to_csv(OUTPUT_CSV, index=False)
        print(f"\nData saved to {OUTPUT_CSV}")

        # Show comprehensive results
        print(f"\n=== DETECTION RESULTS ===")
        print(f"Total posts with movies collected: {len(df)}")

        if len(df) > 0:
            # Movie statistics - all posts now have movies
            all_movies = []
            for movies_list in df['All_Movies']:
                if isinstance(movies_list, list):
                    all_movies.extend(movies_list)

            if all_movies:
                movie_counts = Counter(all_movies)
                print(f"\nTop 20 detected movies:")
                for movie, count in movie_counts.most_common(20):
                    print(f"  {movie}: {count}")

            # Show distribution by sort method
            if 'Sort_Method' in df.columns:
                sort_distribution = df['Sort_Method'].value_counts()
                print(f"\nPosts by source:")
                for method, count in sort_distribution.items():
                    print(f"  {method}: {count}")

            # Show sample detections with emotions
            print(f"\nSample movie detections with emotions:")
            sample = df[['Title', 'Primary_Movie', 'Movie_Count', 'emotion_1', 'emotion_1_score']].head(15)
            for idx, row in sample.iterrows():
                title_short = row['Title'][:60] + '...' if len(row['Title']) > 60 else row['Title']
                print(f"  '{title_short}' -> {row['Primary_Movie']} | {row['emotion_1']} ({row['emotion_1_score']:.3f})")

            # Show emotion statistics
            print(f"\nTop emotions detected:")
            all_emotions = list(df['emotion_1']) + list(df['emotion_2']) + list(df['emotion_3'])
            emotion_counts = Counter(all_emotions)
            for emotion, count in emotion_counts.most_common(12):
                print(f"  {emotion}: {count}")

            # Show posts with multiple movies
            multi_movie_df = df[df['Movie_Count'] > 1]
            print(f"\nPosts with multiple movies detected: {len(multi_movie_df)}")
            print(f"Average movies per post: {df['Movie_Count'].mean():.2f}")
            print(f"Maximum movies in single post: {df['Movie_Count'].max()}")

            # Show time distribution
            if 'Timestamp_UTC' in df.columns:
                df['Hour'] = pd.to_datetime(df['Timestamp_UTC']).dt.hour
                hour_dist = df['Hour'].value_counts().head(5)
                print(f"\nTop posting hours (UTC):")
                for hour, count in hour_dist.items():
                    print(f"  {hour}:00 - {count} posts")

            # Quality metrics
            avg_content_length = df['Content'].str.len().mean()
            posts_with_comments = len(df[df['Comments_Sample'].str.len() > 10])

            print(f"\n=== QUALITY METRICS ===")
            print(f"Average content length: {avg_content_length:.0f} characters")
            print(f"Posts with meaningful comments: {posts_with_comments}")
            print(f"Unique movies detected: {len(set(all_movies))}")

        print(f"\n=== ANALYSIS COMPLETE ===")
        print(f"Dataset contains {len(df)} posts, all with detected movies")
        print(f"Data includes movie detection + top 3 emotions with scores for each post")
        print(f"Mixed data sources (hot/new/top) for better diversity")
        print(f"No 'Unknown' entries - only posts with successfully detected movies are included")
        print(f"File saved as: {OUTPUT_CSV}")

        # Additional analysis suggestions
        if len(df) >= 1000:
            print(f"\n=== SUCCESS! ===")
            print(f"✓ Successfully collected {len(df)} posts with movie mentions")
            print(f"✓ Ready for sentiment analysis and machine learning")
            print(f"✓ Comprehensive emotion data available")
            print(f"✓ Optimized for TrueFilm community discussions")
        else:
            print(f"\n=== PARTIAL SUCCESS ===")
            print(f"⚠ Collected {len(df)} posts (target was 1000)")
            print(f"⚠ Consider running again or adjusting detection parameters")

    except Exception as e:
        print(f"Script failed: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

=== Reddit TrueFilm Movie Sentiment Analysis - 1000 Posts ===
Initializing enhanced movie detection system for TrueFilm...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.



Added 370 patterns to EntityRuler
Detection system ready with 370 movie patterns

Scraping 1000 posts WITH movies from r/TrueFilm...
Will scan up to 3000 posts to find 1000 with detected movies

Fetching hot posts...


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 25/1000 posts with movies | Scanned 28 total | Skipped 3


...

Progress: 950/1000 posts with movies | Scanned 1110 total | Skipped 160


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 975/1000 posts with movies | Scanned 1143 total | Skipped 168


It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/latest/getting_started/multiple_instances.html#discord-bots-and-asynchronous-environments for more info.

It is strongly recommended to use Async PRAW: https://asyncpraw.readthedocs.io.
See https://praw.readthedocs.io/en/l

Progress: 1000/1000 posts with movies | Scanned 1175 total | Skipped 175

Scraping complete!
Posts scanned: 1176
Posts with movies found: 1000
Posts skipped (no movies): 175
Success rate: 85.0%

Data saved to r_truefilm_1000_posts.csv

=== DETECTION RESULTS ===
Total posts with movies collected: 1000

Top 20 detected movies:
  Up: 681
  Captain America: 337
  Saw: 251
  Aquaman: 243
  Favourite: 71
  Alien: 45
  Iron Man: 35
  2001 A Space Odyssey: 35
  Persona: 33
  Burning: 26
  Star Wars: 24
  Parasite: 23
  Psycho: 22
  The Godfather: 22
  Get Out: 21
  Superman: 21
  Joker: 20
  Mulholland Drive: 19
  The Dark Knight: 18
  Scream: 18

Posts by source:
  hot: 797
  new: 203

Sample movie detections with emotions:
  '"Eddington" (2025) - Both Sides are Bad, But One Side is Muc...' -> Civil War | anger (0.746)
  'Looking for movies with a similar feel to Monogatari Series' -> Pierrot Le Fou | love (1.073)
  'What Have You Been Watching? (Week of (August 10, 2025)' -> Skyfall | joy (1

### ***Visualization***

In [ ]:
import pandas as pd
from collections import Counter
import ast

# Load the CSV
df = pd.read_csv("r_truefilm_1000_posts.csv")

# Convert each "All_Movies" entry to a list
def convert_to_list(cell):
    if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == '[]':
        return []
    try:
        # Use ast.literal_eval to safely parse the string representation of a list
        return [movie.strip().lower() for movie in ast.literal_eval(cell)]
    except (ValueError, SyntaxError):
        print(f"Warning: Could not parse '{cell}'. Skipping this row.")
        return []

df["All_Movies"] = df["All_Movies"].apply(convert_to_list)

# Count and sort movie mentions
movie_counter = Counter()
for movie_list in df["All_Movies"]:
    movie_counter.update(movie_list)

# Sort by frequency (descending)
movie_counts = dict(sorted(movie_counter.items(), key=lambda item: item[1], reverse=True))

# Print each movie and its count
for movie, count in movie_counts.items():
    print(f"{movie}: {count}")

up: 681
captain america: 337
saw: 251
aquaman: 243
favourite: 71
alien: 45
iron man: 35
2001 a space odyssey: 35
persona: 33
burning: 26
star wars: 24
parasite: 23
psycho: 22
the godfather: 22
get out: 21
superman: 21
joker: 20
mulholland drive: 19
the dark knight: 18
scream: 18
civil war: 17
batman: 17
marvel cinematic universe: 17
taxi driver: 17
contempt: 15
solo: 15
godfather: 15
rocky: 14
mission impossible: 14
there will be blood: 14
dune: 12
jaws: 11
spider-man: 11
endgame: 11
predator: 10
the shining: 10
lord of the rings: 10
la dolce vita: 10
terminator: 10
blade runner: 10
avengers: 9
jurassic park: 9
amour: 8
green lantern: 8
hereditary: 8
the matrix: 8
infinity war: 8
inception: 8
bicycle thieves: 8
breathless: 8
mad max: 8
the 400 blows: 7
halloween: 7
portrait of a lady on fire: 7
citizen kane: 7
black panther: 7
midsommar: 7
vertigo: 7
flash: 7
goodfellas: 7
raging bull: 7
manhattan: 7
titanic: 6
john wick: 6
guardians of the galaxy: 6
batman begins: 6
dark knight: 6
the

**Top 10 movies in r/TrueFilm**

In [ ]:
import matplotlib.pyplot as plt

# Choose how many top movies to show
TOP_N = 10

# Get top N items
top_movies = list(movie_counts.items())[:TOP_N]
movie_names, counts = zip(*top_movies)

# Plotting
plt.figure(figsize=(12, 6))
plt.barh(movie_names[::-1], counts[::-1], color='lavender')  # reverse for descending top-down
plt.xlabel("Mention Count")
plt.title(f"Top {TOP_N} Most Mentioned Movies in r/truefilm")
plt.tight_layout()
plt.show()

**Average Sentiment scores per Day r/truefilm**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Load the data
df = pd.read_csv("r_letterboxd_1000_posts.csv")

# Extract date from 'Timestamp_UTC'
df['Date'] = pd.to_datetime(df['Timestamp_UTC'], errors='coerce').dt.date

# Parse emotions from individual columns (emotion_1, emotion_2, emotion_3 with their scores)
def parse_emotions_from_columns(row):
    emotions = {}
    # Get emotion scores from the separate columns
    emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']
    score_columns = ['emotion_1_score', 'emotion_2_score', 'emotion_3_score']

    for i, (emo_col, score_col) in enumerate(zip(emotion_columns, score_columns)):
        if emo_col in row and score_col in row:
            if pd.notna(row[emo_col]) and pd.notna(row[score_col]):
                emotion = str(row[emo_col]).lower()
                score = float(row[score_col])
                emotions[emotion] = score
    return emotions

# Apply parsing and expand to new columns
emotion_df = df.apply(parse_emotions_from_columns, axis=1).apply(pd.Series)
emotion_df['Date'] = df['Date']

# Group by date and average
daily_sentiments = emotion_df.groupby('Date').mean().reset_index()

# Melt for seaborn
melted = daily_sentiments.melt(id_vars='Date', var_name='Emotion', value_name='Score')

# Plot
plt.figure(figsize=(16, 6))
sns.lineplot(data=melted, x='Date', y='Score', hue='Emotion', marker='o')
plt.title("Average Sentiment Scores per Day in r/TrueFilm", fontsize=16)
plt.xlabel("Date", fontsize=12)
plt.ylabel("Average Sentiment Score", fontsize=12)
plt.xticks(rotation=45)
plt.legend(title="Sentiment", fontsize=10, title_fontsize=12)
plt.tight_layout()
plt.show()

# COMBINED

**Average Emotion Scores per Subreddit**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# 📥 Load and combine data
files = {
    "r_movies_1000_posts.csv": "r/movies",
    "r_letterboxd_1000_posts.csv": "r/letterboxd",
    "r_truefilm_1000_posts.csv": "r/truefilm",
}

records = []

for fp, label in files.items():
    try:
        df = pd.read_csv(fp)
        print(f"Loaded {fp} - Shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")

        # Extract emotions from the emotion columns (emotion_1, emotion_2, emotion_3)
        emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']

        for _, row in df.iterrows():
            for col in emotion_columns:
                if col in df.columns and pd.notna(row[col]):
                    emotion = str(row[col]).strip().lower()
                    if emotion and emotion != 'neutral':  # Skip neutral emotions for cleaner analysis
                        records.append({"dataset": label, "emotion": emotion})

        print(f"Extracted {sum(1 for r in records if r['dataset'] == label)} emotions from {label}")

    except FileNotFoundError:
        print(f"Warning: {fp} not found. Skipping...")
    except Exception as e:
        print(f"Error processing {fp}: {e}")

if not records:
    print("No emotion data found. Please check your CSV files and column names.")
    # Let's create some sample data for demonstration
    sample_emotions = ['joy', 'excitement', 'love', 'disappointment', 'appreciation']
    for dataset in files.values():
        for emotion in sample_emotions:
            for _ in range(20):  # 20 instances of each emotion per dataset
                records.append({"dataset": dataset, "emotion": emotion})
    print("Created sample data for demonstration.")

# Convert to DataFrame
em = pd.DataFrame(records)
print(f"\nTotal emotion records: {len(em)}")
print(f"Unique emotions: {em['emotion'].nunique()}")
print(f"Datasets: {em['dataset'].unique()}")

# Count emotions by dataset
counts = em.groupby(['dataset', 'emotion']).size().reset_index(name='count')

# Filter to top 5 emotions overall
top5_emotions = counts.groupby('emotion')['count'].sum().nlargest(5).index
subset = counts[counts['emotion'].isin(top5_emotions)]

print(f"\nTop 5 emotions: {list(top5_emotions)}")

#  Plot grouped bar chart
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=subset, x='emotion', y='count', hue='dataset', ci=None, palette='tab10')

#  Add count labels on top of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%d', label_type='edge')

ax.set_title("Top 5 Emotion Counts per Subreddit", fontsize=14, fontweight='bold')
ax.set_ylabel("Count of Emotions", fontsize=12)
ax.set_xlabel("Emotion", fontsize=12)
plt.legend(title="Subreddits", title_fontsize=10)
sns.despine(left=True)
plt.tight_layout()
plt.xticks(rotation=45)
plt.show()

#  Additional analysis
print("\n=== EMOTION ANALYSIS SUMMARY ===")

# Overall emotion distribution
overall_counts = em['emotion'].value_counts().head(10)
print("\nTop 10 emotions across all datasets:")
for emotion, count in overall_counts.items():
    print(f"  {emotion}: {count}")

# Dataset-specific analysis
print("\nEmotion distribution by dataset:")
for dataset in em['dataset'].unique():
    dataset_emotions = em[em['dataset'] == dataset]['emotion'].value_counts().head(5)
    print(f"\n{dataset} - Top 5 emotions:")
    for emotion, count in dataset_emotions.items():
        print(f"  {emotion}: {count}")

# Cross-dataset comparison
pivot_table = em.pivot_table(index='emotion', columns='dataset', values='emotion', aggfunc='count', fill_value=0)
print(f"\nCross-dataset emotion comparison:")
print(pivot_table.head(10))

Loaded r_movies_1000_posts.csv - Shape: (1000, 15)
Columns: ['Title', 'Author', 'All_Movies', 'Primary_Movie', 'Movie_Count', 'Content', 'Comments_Sample', 'Timestamp_UTC', 'Sort_Method', 'emotion_1', 'emotion_1_score', 'emotion_2', 'emotion_2_score', 'emotion_3', 'emotion_3_score']
Extracted 2019 emotions from r/movies
Loaded r_letterboxd_1000_posts.csv - Shape: (1000, 15)
Columns: ['Title', 'Author', 'All_Movies', 'Primary_Movie', 'Movie_Count', 'Content', 'Comments_Sample', 'Timestamp_UTC', 'Sort_Method', 'emotion_1', 'emotion_1_score', 'emotion_2', 'emotion_2_score', 'emotion_3', 'emotion_3_score']
Extracted 2008 emotions from r/letterboxd
Loaded r_truefilm_1000_posts.csv - Shape: (1000, 15)
Columns: ['Title', 'Author', 'All_Movies', 'Primary_Movie', 'Movie_Count', 'Content', 'Comments_Sample', 'Timestamp_UTC', 'Sort_Method', 'emotion_1', 'emotion_1_score', 'emotion_2', 'emotion_2_score', 'emotion_3', 'emotion_3_score']
Extracted 2113 emotions from r/truefilm

Total emotion records

/tmp/ipython-input-433110046.py:65: FutureWarning: 

The `ci` parameter is deprecated. Use `errorbar=None` for the same effect.

  ax = sns.barplot(data=subset, x='emotion', y='count', hue='dataset', ci=None, palette='tab10')


**Emotions co-occurence HeatMap**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations
from collections import Counter
import numpy as np

# 🎯 File paths and subreddit mapping
files = {
    "r_movies_1000_posts.csv": "r/movies",
    "r_letterboxd_1000_posts.csv": "r/letterboxd",
    "r_truefilm_1000_posts.csv": "r/truefilm",
}

# 🔄 Function to extract emotion co-occurrences from individual emotion columns
def extract_cooccurrence(df):
    cooc = Counter()

    # Check what emotion columns exist in the dataframe
    emotion_cols = [col for col in df.columns if col.startswith('emotion_') and not col.endswith('_score')]
    print(f"Found emotion columns: {emotion_cols}")

    for idx, row in df.iterrows():
        emotions = []

        # Extract emotions from the individual columns
        for col in emotion_cols:
            if col in row and pd.notna(row[col]):
                emotion = str(row[col]).strip().lower()
                if emotion and emotion != 'neutral':  # Skip neutral emotions
                    emotions.append(emotion)

        # Remove duplicates and create combinations
        emotions = sorted(set(emotions))

        # Only create pairs if we have at least 2 different emotions
        if len(emotions) >= 2:
            for pair in combinations(emotions, 2):
                cooc[pair] += 1

    return cooc

# 🔢 Aggregate co-occurrences per subreddit
subreddit_coocs = {}

for file, subreddit in files.items():
    try:
        print(f"\n📊 Processing {file} for {subreddit}...")
        df = pd.read_csv(file)

        # Print first few rows to debug
        print(f"Columns in {file}: {list(df.columns)}")
        print(f"Shape: {df.shape}")

        # Show sample of emotion data
        emotion_sample_cols = [col for col in df.columns if 'emotion' in col.lower()]
        if emotion_sample_cols:
            print("Sample emotion data:")
            print(df[emotion_sample_cols].head(3))

        cooc = extract_cooccurrence(df)
        subreddit_coocs[subreddit] = cooc

        print(f"Found {len(cooc)} emotion pairs in {subreddit}")

    except FileNotFoundError:
        print(f"[⚠️] File not found: {file}")
        continue
    except Exception as e:
        print(f"[❌] Error processing {file}: {e}")
        continue

# 🎨 Plot heatmaps
for subreddit, cooc in subreddit_coocs.items():
    if not cooc:
        print(f"[⚠️] No co-occurrence data for {subreddit}")
        continue

    print(f"\n🎨 Creating heatmap for {subreddit}...")

    # Get all unique emotions
    unique_emotions = sorted(set(e for pair in cooc for e in pair))
    print(f"Unique emotions found: {unique_emotions}")

    # Create matrix
    matrix = pd.DataFrame(0, index=unique_emotions, columns=unique_emotions)

    # Fill the matrix
    for (emo1, emo2), count in cooc.items():
        matrix.at[emo1, emo2] = count
        matrix.at[emo2, emo1] = count  # symmetric

    # Plot
    plt.figure(figsize=(12, 10))
    sns.heatmap(matrix, annot=True, fmt='d', cmap="Reds", linewidths=0.5, square=True, cbar_kws={'label': 'Co-occurrence Count'})
    plt.title(f"Emotion Co-occurrence Heatmap - {subreddit}", fontsize=16, fontweight='bold')
    plt.xlabel('Emotions', fontsize=12)
    plt.ylabel('Emotions', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Print top co-occurrences
    if cooc:
        print(f"\nTop 10 emotion pairs in {subreddit}:")
        for pair, count in cooc.most_common(10):
            print(f"  {pair[0]} + {pair[1]}: {count}")

# 📈 Summary statistics
print("\n" + "="*60)
print("📈 SUMMARY STATISTICS")
print("="*60)

total_pairs = sum(len(cooc) for cooc in subreddit_coocs.values())
print(f"Total unique emotion pairs across all subreddits: {total_pairs}")

for subreddit, cooc in subreddit_coocs.items():
    if cooc:
        total_cooc = sum(cooc.values())
        unique_pairs = len(cooc)
        print(f"\n{subreddit}:")
        print(f"  - Unique emotion pairs: {unique_pairs}")
        print(f"  - Total co-occurrences: {total_cooc}")
        print(f"  - Most common pair: {cooc.most_common(1)[0] if cooc else 'None'}")


📊 Processing r_movies_1000_posts.csv for r/movies...
Columns in r_movies_1000_posts.csv: ['Title', 'Author', 'All_Movies', 'Primary_Movie', 'Movie_Count', 'Content', 'Comments_Sample', 'Timestamp_UTC', 'Sort_Method', 'emotion_1', 'emotion_1_score', 'emotion_2', 'emotion_2_score', 'emotion_3', 'emotion_3_score']
Shape: (1000, 15)
Sample emotion data:
  emotion_1  emotion_1_score  emotion_2  emotion_2_score  emotion_3  \
0   neutral            0.852  confusion            0.271        joy   
1       joy            1.020    neutral            0.843  confusion   
2   neutral            0.885        joy            0.629  confusion   

   emotion_3_score  
0            0.130  
1            0.262  
2            0.229  
Found emotion columns: ['emotion_1', 'emotion_2', 'emotion_3']
Found 43 emotion pairs in r/movies

📊 Processing r_letterboxd_1000_posts.csv for r/letterboxd...
Columns in r_letterboxd_1000_posts.csv: ['Title', 'Author', 'All_Movies', 'Primary_Movie', 'Movie_Count', 'Content', '

# *Dashboard*

In [ ]:
!pip install gradio

In [ ]:
# Complete Movie Subreddit Insights Dashboard - Library Version
# This notebook creates an interactive dashboard for analyzing movie discussions across Reddit
# Uses built-in libraries instead of requiring file uploads

# ============================================================================
# SECTION 1: INSTALLATION AND IMPORTS
# ============================================================================

# Install required packages
!pip install gradio pandas matplotlib seaborn numpy scikit-learn

# Import libraries
import gradio as gr
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re
from itertools import combinations
import random
from datetime import datetime, timedelta

# Set matplotlib backend for Colab
plt.switch_backend('Agg')
sns.set_style("whitegrid")

# ============================================================================
# SECTION 2: SAMPLE DATA GENERATION
# ============================================================================

# Sample movie data for demonstration
SAMPLE_MOVIES = [
    "the batman", "dune", "spider-man", "avengers", "inception", "joker",
    "parasite", "la la land", "moonlight", "blade runner 2049", "mad max fury road",
    "the godfather", "pulp fiction", "the dark knight", "goodfellas", "casablanca",
    "citizen kane", "vertigo", "2001 a space odyssey", "mulholland drive",
    "there will be blood", "no country for old men", "her", "whiplash", "birdman"
]

EMOTIONS = [
    "joy", "sadness", "anger", "fear", "surprise", "disgust", "trust", "anticipation",
    "love", "excitement", "nostalgia", "confusion", "disappointment", "admiration"
]

def generate_sample_data(subreddit_name, num_posts=1000):
    """Generate sample data that mimics the structure of your CSV files"""
    data = []

    # Generate posts over the last 30 days
    start_date = datetime.now() - timedelta(days=30)

    for i in range(num_posts):
        # Random timestamp
        timestamp = start_date + timedelta(
            days=random.randint(0, 30),
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )

        # Random movie mentions (0-5 movies per post)
        num_movies = random.choices([0, 1, 2, 3, 4, 5], weights=[20, 40, 25, 10, 3, 2])[0]
        movies = random.sample(SAMPLE_MOVIES, min(num_movies, len(SAMPLE_MOVIES)))

        # Generate emotions with scores
        num_emotions = random.choices([1, 2, 3], weights=[50, 35, 15])[0]
        selected_emotions = random.sample(EMOTIONS, num_emotions)

        # Adjust emotion probabilities based on subreddit
        if 'movies' in subreddit_name.lower():
            # r/movies tends to have more mainstream emotions
            emotion_weights = [0.8 if e in ['joy', 'excitement', 'love', 'anger'] else 0.2 for e in selected_emotions]
        elif 'letterboxd' in subreddit_name.lower():
            # r/letterboxd tends to be more analytical
            emotion_weights = [0.8 if e in ['admiration', 'nostalgia', 'love', 'disappointment'] else 0.3 for e in selected_emotions]
        elif 'truefilm' in subreddit_name.lower():
            # r/truefilm tends to be more intellectual
            emotion_weights = [0.8 if e in ['admiration', 'confusion', 'anticipation', 'trust'] else 0.2 for e in selected_emotions]
        else:
            emotion_weights = [1.0] * len(selected_emotions)

        # Normalize weights
        total_weight = sum(emotion_weights)
        emotion_weights = [w/total_weight for w in emotion_weights]

        row = {
            'Timestamp_UTC': timestamp.isoformat(),
            'All_Movies': str(movies),  # Store as string representation of list
        }

        # Add emotion columns
        for j in range(3):
            if j < len(selected_emotions):
                row[f'emotion_{j+1}'] = selected_emotions[j]
                # Generate scores with some bias based on emotion weights
                base_score = random.uniform(0.3, 0.9)
                weighted_score = base_score * emotion_weights[j] * random.uniform(0.7, 1.3)
                row[f'emotion_{j+1}_score'] = min(1.0, max(0.0, weighted_score))
            else:
                row[f'emotion_{j+1}'] = None
                row[f'emotion_{j+1}_score'] = None

        data.append(row)

    return pd.DataFrame(data)

# Generate sample datasets
print(" Generating sample datasets...")

SUBREDDIT_DATA = {
    "r/movies": generate_sample_data("r/movies"),
    "r/letterboxd": generate_sample_data("r/letterboxd"),
    "r/truefilm": generate_sample_data("r/truefilm")
}

print(" Sample data generated successfully!")
print(f" Each dataset contains {len(SUBREDDIT_DATA['r/movies'])} posts")

# ============================================================================
# SECTION 3: HELPER FUNCTIONS (Updated for DataFrame input)
# ============================================================================

def process_and_plot_movie_mentions(df, subreddit_name, top_n=10):
    """Generate movie mentions plot from DataFrame"""
    fig = plt.figure(figsize=(12, 6))
    try:
        # Convert All_Movies to list format
        def convert_to_list(cell):
            if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == '[]':
                return []
            try:
                # Remove quotes and brackets, split by comma
                clean_cell = cell.strip("[]").replace("'", "").replace('"', '')
                if not clean_cell:
                    return []
                return [movie.strip().lower() for movie in clean_cell.split(',') if movie.strip()]
            except:
                return []

        df["All_Movies_List"] = df["All_Movies"].apply(convert_to_list)

        # Count movie mentions
        movie_counter = Counter()
        for movie_list in df["All_Movies_List"]:
            movie_counter.update(movie_list)

        # Sort by frequency
        movie_counts = dict(sorted(movie_counter.items(), key=lambda item: item[1], reverse=True))

        # Get top N movies
        top_movies = list(movie_counts.items())[:top_n]

        if not top_movies:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, f"No movie mentions found in {subreddit_name} data.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            return fig

        movie_names, counts = zip(*top_movies)

        # Choose colors based on subreddit
        if 'movies' in subreddit_name.lower():
            color = 'lightcoral'
        elif 'letterboxd' in subreddit_name.lower():
            color = 'lightblue'
        elif 'truefilm' in subreddit_name.lower():
            color = 'lightgreen'
        else:
            color = 'lavender'

        # Create horizontal bar chart
        ax = fig.add_subplot(111)
        ax.barh(movie_names[::-1], counts[::-1], color=color)
        ax.set_xlabel("Mention Count", fontsize=12)
        ax.set_title(f"Top {top_n} Most Mentioned Movies in {subreddit_name}", fontsize=14)
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"An error occurred: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

def process_and_plot_sentiment(df, subreddit_name):
    """Generate sentiment timeline plot from DataFrame"""
    fig = plt.figure(figsize=(16, 6))
    try:
        # Extract date from 'Timestamp_UTC'
        df['Date'] = pd.to_datetime(df['Timestamp_UTC'], errors='coerce').dt.date

        # Parse emotions from individual columns
        def parse_emotions_from_columns(row):
            emotions = {}
            emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']
            score_columns = ['emotion_1_score', 'emotion_2_score', 'emotion_3_score']

            for i, (emo_col, score_col) in enumerate(zip(emotion_columns, score_columns)):
                if emo_col in row and score_col in row:
                    if pd.notna(row[emo_col]) and pd.notna(row[score_col]):
                        emotion = str(row[emo_col]).lower()
                        score = float(row[score_col])
                        emotions[emotion] = score
            return emotions

        # Apply parsing and expand to new columns
        emotion_df = df.apply(parse_emotions_from_columns, axis=1).apply(pd.Series)
        emotion_df['Date'] = df['Date']

        # Drop columns that are all NaN
        emotion_df = emotion_df.dropna(axis=1, how='all')

        if emotion_df.empty or 'Date' not in emotion_df.columns or len(emotion_df.columns) <= 1:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, f"No valid emotion data found for sentiment analysis in {subreddit_name}.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            return fig

        # Group by date and average
        daily_sentiments = emotion_df.groupby('Date').mean().reset_index()

        # Melt for seaborn
        melted = daily_sentiments.melt(id_vars='Date', var_name='Emotion', value_name='Score')

        ax = fig.add_subplot(111)
        sns.lineplot(data=melted, x='Date', y='Score', hue='Emotion', marker='o', ax=ax)
        ax.set_title(f"Average Sentiment Scores per Day in {subreddit_name}", fontsize=16)
        ax.set_xlabel("Date", fontsize=12)
        ax.set_ylabel("Average Sentiment Score", fontsize=12)
        plt.xticks(rotation=45)
        ax.legend(title="Sentiment", fontsize=10, title_fontsize=12)
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"An error occurred: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

def generate_movie_specific_sentiment_plot(movie_name, df, subreddit_name):
    """Generate movie-specific sentiment plot from DataFrame"""
    fig = plt.figure(figsize=(14, 6))
    try:
        # Convert timestamp to datetime
        df['Timestamp_UTC'] = pd.to_datetime(df['Timestamp_UTC'], errors='coerce')
        df.dropna(subset=['Timestamp_UTC'], inplace=True)

        # Convert All_Movies to list format
        def convert_to_list(cell):
            if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == '[]':
                return []
            try:
                clean_cell = cell.strip("[]").replace("'", "").replace('"', '')
                if not clean_cell:
                    return []
                return [movie.strip().lower() for movie in clean_cell.split(',') if movie.strip()]
            except:
                return []

        df['All_Movies_List'] = df['All_Movies'].apply(convert_to_list)

        # Filter rows that mention the selected movie
        filtered_df = df[df['All_Movies_List'].apply(
            lambda x: any(movie_name.strip().lower() in movie.lower() for movie in x)
        )].copy()

        if filtered_df.empty:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, f"No mentions found for '{movie_name}' in {subreddit_name} data.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='red')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            return fig

        # Extract emotions and scores
        rows = []
        emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']
        score_columns = ['emotion_1_score', 'emotion_2_score', 'emotion_3_score']

        for _, row in filtered_df.iterrows():
            for emo_col, score_col in zip(emotion_columns, score_columns):
                if pd.notna(row.get(emo_col)) and pd.notna(row.get(score_col)):
                    rows.append({
                        'Date': row['Timestamp_UTC'].date(),
                        'Emotion': str(row[emo_col]).lower(),
                        'Score': float(row[score_col])
                    })

        if not rows:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, f"No emotion data found for '{movie_name}' in {subreddit_name}.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            return fig

        emotion_df = pd.DataFrame(rows)

        # Pivot to get each emotion as a column
        pivot_df = emotion_df.pivot_table(index='Date', columns='Emotion', values='Score', aggfunc='mean').fillna(0)

        ax = fig.add_subplot(111)

        for col in pivot_df.columns:
            sns.lineplot(data=pivot_df, x=pivot_df.index, y=col, label=col, marker='o', ax=ax)

        ax.set_title(f"Daily Emotion Trends for '{movie_name.title()}' Mentions in {subreddit_name}")
        ax.set_xlabel("Date")
        ax.set_ylabel("Average Emotion Score")
        plt.xticks(rotation=45)
        ax.legend(title="Emotions", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"An unexpected error occurred: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

def process_and_plot_cooccurrence_heatmap(df, subreddit_name):
    """Generate emotion co-occurrence heatmap from DataFrame"""
    fig = plt.figure(figsize=(10, 8))
    try:
        # Function to extract emotion co-occurrences
        def extract_cooccurrence(df_sub):
            cooc = Counter()
            emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']

            for _, row in df_sub.iterrows():
                emotions = []
                for col in emotion_columns:
                    if pd.notna(row.get(col)):
                        emotions.append(str(row[col]).lower())

                # Remove duplicates and get pairs
                emotions = sorted(set(emotions))
                for pair in combinations(emotions, 2):
                    cooc[pair] += 1
            return cooc

        cooc = extract_cooccurrence(df)

        if not cooc:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, f"No co-occurrence data for {subreddit_name}.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            return fig

        # Get all unique emotions
        unique_emotions = sorted(set(e for pair in cooc for e in pair))
        matrix = pd.DataFrame(0, index=unique_emotions, columns=unique_emotions)

        # Fill the matrix
        for (emo1, emo2), count in cooc.items():
            matrix.at[emo1, emo2] = count
            matrix.at[emo2, emo1] = count  # symmetric

        ax = fig.add_subplot(111)

        # Choose colormap based on subreddit
        if 'movies' in subreddit_name.lower():
            cmap = "Reds"
        elif 'letterboxd' in subreddit_name.lower():
            cmap = "Blues"
        elif 'truefilm' in subreddit_name.lower():
            cmap = "Greens"
        else:
            cmap = "Purples"

        sns.heatmap(matrix, annot=True, cmap=cmap, linewidths=0.5, square=True, ax=ax)
        ax.set_title(f"Emotion Co-occurrence Heatmap - {subreddit_name}")
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"An error occurred: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

# ============================================================================
# SECTION 4: SUBREDDIT-SPECIFIC FUNCTIONS (Updated)
# ============================================================================

# Functions for r/movies
def generate_rmovies_mentions_plot():
    return process_and_plot_movie_mentions(SUBREDDIT_DATA["r/movies"], "r/movies")

def generate_rmovies_sentiment_plot():
    return process_and_plot_sentiment(SUBREDDIT_DATA["r/movies"], "r/movies")

def generate_rmovies_specific_sentiment_plot(movie_name):
    return generate_movie_specific_sentiment_plot(movie_name, SUBREDDIT_DATA["r/movies"], "r/movies")

def generate_rmovies_cooccurrence_plot():
    return process_and_plot_cooccurrence_heatmap(SUBREDDIT_DATA["r/movies"], "r/movies")

# Functions for r/letterboxd
def generate_letterboxd_mentions_plot():
    return process_and_plot_movie_mentions(SUBREDDIT_DATA["r/letterboxd"], "r/letterboxd")

def generate_letterboxd_sentiment_plot():
    return process_and_plot_sentiment(SUBREDDIT_DATA["r/letterboxd"], "r/letterboxd")

def generate_letterboxd_specific_sentiment_plot(movie_name):
    return generate_movie_specific_sentiment_plot(movie_name, SUBREDDIT_DATA["r/letterboxd"], "r/letterboxd")

def generate_letterboxd_cooccurrence_plot():
    return process_and_plot_cooccurrence_heatmap(SUBREDDIT_DATA["r/letterboxd"], "r/letterboxd")

# Functions for r/truefilm
def generate_truefilm_mentions_plot():
    return process_and_plot_movie_mentions(SUBREDDIT_DATA["r/truefilm"], "r/truefilm")

def generate_truefilm_sentiment_plot():
    return process_and_plot_sentiment(SUBREDDIT_DATA["r/truefilm"], "r/truefilm")

def generate_truefilm_specific_sentiment_plot(movie_name):
    return generate_movie_specific_sentiment_plot(movie_name, SUBREDDIT_DATA["r/truefilm"], "r/truefilm")

def generate_truefilm_cooccurrence_plot():
    return process_and_plot_cooccurrence_heatmap(SUBREDDIT_DATA["r/truefilm"], "r/truefilm")

# ============================================================================
# SECTION 5: COMPARISON FUNCTIONS (Updated)
# ============================================================================

def generate_all_subreddits_average_emotion_plot():
    """Generate combined average emotion scores plot"""
    records = []
    fig = plt.figure(figsize=(14, 8))

    try:
        for subreddit, df in SUBREDDIT_DATA.items():
            emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']
            score_columns = ['emotion_1_score', 'emotion_2_score', 'emotion_3_score']

            for _, row in df.iterrows():
                for emo_col, score_col in zip(emotion_columns, score_columns):
                    if pd.notna(row.get(emo_col)) and pd.notna(row.get(score_col)):
                        emotion = str(row[emo_col]).lower()
                        score = float(row[score_col])
                        records.append({
                            'dataset': subreddit,
                            'emotion': emotion,
                            'score': score
                        })

        if not records:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, "No emotion data found for average scores.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            plt.tight_layout()
            return fig

        combined_df = pd.DataFrame(records)
        avg_scores = combined_df.groupby(['dataset', 'emotion'])['score'].mean().reset_index()

        # Filter to top emotions for better visualization
        top_emotions = combined_df.groupby('emotion')['score'].count().nlargest(8).index
        avg_scores_filtered = avg_scores[avg_scores['emotion'].isin(top_emotions)]

        ax = fig.add_subplot(111)
        sns.barplot(data=avg_scores_filtered, x='emotion', y='score', hue='dataset', errorbar=None, palette='Set1', ax=ax)

        for container in ax.containers:
            ax.bar_label(container, fmt='%.2f', label_type='edge')

        ax.set_title("Average Emotion Scores: r/movies vs r/letterboxd vs r/truefilm", fontsize=16)
        ax.set_ylabel("Average Score")
        ax.set_xlabel("Emotion")
        ax.legend(title="Subreddits")
        plt.xticks(rotation=45, ha='right')
        sns.despine(left=True)
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Unexpected error: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

def generate_movie_popularity_comparison():
    """Generate a comparison of most popular movies across all subreddits"""
    fig = plt.figure(figsize=(16, 10))

    try:
        all_movie_data = {}

        for subreddit, df in SUBREDDIT_DATA.items():
            # Convert All_Movies to list format
            def convert_to_list(cell):
                if pd.isna(cell) or not isinstance(cell, str) or cell.strip() == '[]':
                    return []
                try:
                    clean_cell = cell.strip("[]").replace("'", "").replace('"', '')
                    if not clean_cell:
                        return []
                    return [movie.strip().lower() for movie in clean_cell.split(',') if movie.strip()]
                except:
                    return []

            df["All_Movies_List"] = df["All_Movies"].apply(convert_to_list)

            # Count movie mentions
            movie_counter = Counter()
            for movie_list in df["All_Movies_List"]:
                movie_counter.update(movie_list)

            all_movie_data[subreddit] = dict(movie_counter.most_common(15))

        # Create comparison dataframe
        comparison_records = []
        for subreddit, movies in all_movie_data.items():
            for movie, count in movies.items():
                comparison_records.append({
                    'subreddit': subreddit,
                    'movie': movie.title(),
                    'mentions': count
                })

        if not comparison_records:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, "No movie mention data to display.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            return fig

        comp_df = pd.DataFrame(comparison_records)

        # Get top movies overall
        top_movies_overall = comp_df.groupby('movie')['mentions'].sum().nlargest(12).index
        comp_df_filtered = comp_df[comp_df['movie'].isin(top_movies_overall)]

        ax = fig.add_subplot(111)
        sns.barplot(data=comp_df_filtered, y='movie', x='mentions', hue='subreddit',
                   errorbar=None, palette='viridis', ax=ax)

        ax.set_title("Top Movies Mentioned Across All Subreddits", fontsize=16)
        ax.set_xlabel("Number of Mentions")
        ax.set_ylabel("Movie")
        ax.legend(title="Subreddit", bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Error creating movie comparison: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

def generate_sentiment_distribution_plot():
    """Generate sentiment distribution comparison across subreddits"""
    fig = plt.figure(figsize=(15, 8))

    try:
        sentiment_data = []

        for subreddit, df in SUBREDDIT_DATA.items():
            emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']
            score_columns = ['emotion_1_score', 'emotion_2_score', 'emotion_3_score']

            for _, row in df.iterrows():
                for emo_col, score_col in zip(emotion_columns, score_columns):
                    if pd.notna(row.get(emo_col)) and pd.notna(row.get(score_col)):
                        sentiment_data.append({
                            'subreddit': subreddit,
                            'emotion': str(row[emo_col]).lower(),
                            'score': float(row[score_col])
                        })

        if not sentiment_data:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, "No sentiment data available.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            return fig

        sentiment_df = pd.DataFrame(sentiment_data)

        # Create violin plot for score distributions
        ax = fig.add_subplot(111)
        sns.violinplot(data=sentiment_df, x='subreddit', y='score', hue='subreddit',
                      palette='Set3', inner='box', ax=ax)

        ax.set_title("Distribution of Emotion Scores by Subreddit", fontsize=16)
        ax.set_xlabel("Subreddit")
        ax.set_ylabel("Emotion Score")
        ax.legend().set_visible(False)  # Remove redundant legend
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Error creating sentiment distribution: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

def generate_all_subreddits_emotion_plot():
    """Generate combined emotion plot for all subreddits"""
    records = []
    fig = plt.figure(figsize=(14, 8))

    try:
        for subreddit, df in SUBREDDIT_DATA.items():
            emotion_columns = ['emotion_1', 'emotion_2', 'emotion_3']

            for _, row in df.iterrows():
                for col in emotion_columns:
                    if pd.notna(row.get(col)):
                        emotion = str(row[col]).lower()
                        records.append({"dataset": subreddit, "emotion": emotion})

        if not records:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, "No emotion data found across subreddits.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            plt.tight_layout()
            return fig

        em = pd.DataFrame(records)
        counts = em.groupby(['dataset', 'emotion']).size().reset_index(name='count')

        # Filter to top 8 emotions overall
        top_emotions = counts.groupby('emotion')['count'].sum().nlargest(8).index
        subset = counts[counts['emotion'].isin(top_emotions)]

        if subset.empty:
            ax = fig.add_subplot(111)
            ax.text(0.5, 0.5, "No emotion data to display.",
                    horizontalalignment='center', verticalalignment='center', fontsize=14, color='gray')
            ax.set_xticks([])
            ax.set_yticks([])
            ax.set_frame_on(False)
            plt.tight_layout()
            return fig

        # Plot grouped bar chart
        ax = fig.add_subplot(111)
        sns.barplot(data=subset, x='emotion', y='count', hue='dataset', errorbar=None, palette='Set2', ax=ax)

        # Add count labels on top of bars
        for container in ax.containers:
            ax.bar_label(container, fmt='%d', label_type='edge')

        ax.set_title("Top 8 Emotion Counts: r/movies vs r/letterboxd vs r/truefilm", fontsize=16)
        ax.set_ylabel("Count of Emotions")
        ax.set_xlabel("Emotion")
        ax.legend(title="Subreddits")
        plt.xticks(rotation=45, ha='right')
        sns.despine(left=True)
        plt.tight_layout()
        return fig

    except Exception as e:
        ax = fig.add_subplot(111)
        ax.text(0.5, 0.5, f"Unexpected error: {str(e)}",
                horizontalalignment='center', verticalalignment='center', fontsize=12, color='red')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_frame_on(False)
        return fig

# ============================================================================
# SECTION 6: GRADIO INTERFACE
# ============================================================================

# Create the Gradio interface with enhanced styling
with gr.Blocks(
    title="Movie Subreddit Insights Dashboard",
    theme=gr.themes.Soft(
        primary_hue="blue",
        secondary_hue="cyan",
        neutral_hue="gray"
    )
) as demo:

    # Header
    gr.Markdown(
        """
        #  Movie Subreddit Insights Dashboard

        **Explore popular movie mentions and sentiment analysis across Reddit's film communities**

        This dashboard analyzes discussions from three major movie subreddits using **generated sample data**:
        - **r/movies**: General movie discussions and mainstream cinema
        - **r/letterboxd**: Film enthusiasts and reviews
        - **r/truefilm**: In-depth film analysis and art house cinema

         **Note**: This version uses realistic sample data generated from built-in libraries instead of requiring CSV uploads.

        ---
        """
    )

    with gr.Tabs() as tabs:

        # r/movies Tab
        with gr.TabItem(" r/movies", id=0):
            gr.Markdown("### Top Movie Mentions in r/movies")
            gr.Plot(generate_rmovies_mentions_plot, label="Top Movie Mentions")

            gr.Markdown("---")
            gr.Markdown("### Average Sentiment Scores per Day in r/movies")
            gr.Plot(generate_rmovies_sentiment_plot, label="Daily Sentiment Scores")

            gr.Markdown("---")
            gr.Markdown("### Daily Emotion Trends for a Specific Movie in r/movies")
            with gr.Row():
                movie_input_movies = gr.Textbox(
                    label="Enter a Movie Name",
                    placeholder="e.g., 'batman', 'avengers', 'inception'...",
                    info="Search for emotion trends of a specific movie"
                )
            movie_sentiment_plot_movies = gr.Plot(label="Emotion Trends for Specific Movie")
            movie_input_movies.submit(
                generate_rmovies_specific_sentiment_plot,
                inputs=[movie_input_movies],
                outputs=[movie_sentiment_plot_movies]
            )

            gr.Markdown("---")
            gr.Markdown("### Emotion Co-occurrence Heatmap - r/movies")
            gr.Plot(generate_rmovies_cooccurrence_plot, label="Emotion Co-occurrence Heatmap")

        # r/letterboxd Tab
        with gr.TabItem(" r/letterboxd", id=1):
            gr.Markdown("### Top Movie Mentions in r/letterboxd")
            gr.Plot(generate_letterboxd_mentions_plot, label="Top Movie Mentions")

            gr.Markdown("---")
            gr.Markdown("### Average Sentiment Scores per Day in r/letterboxd")
            gr.Plot(generate_letterboxd_sentiment_plot, label="Daily Sentiment Scores")

            gr.Markdown("---")
            gr.Markdown("### Daily Emotion Trends for a Specific Movie in r/letterboxd")
            with gr.Row():
                movie_input_letterboxd = gr.Textbox(
                    label="Enter a Movie Name",
                    placeholder="e.g., 'mulholland drive', 'parasite', 'moonlight'...",
                    info="Search for emotion trends of a specific movie"
                )
            movie_sentiment_plot_letterboxd = gr.Plot(label="Emotion Trends for Specific Movie")
            movie_input_letterboxd.submit(
                generate_letterboxd_specific_sentiment_plot,
                inputs=[movie_input_letterboxd],
                outputs=[movie_sentiment_plot_letterboxd]
            )

            gr.Markdown("---")
            gr.Markdown("### Emotion Co-occurrence Heatmap - r/letterboxd")
            gr.Plot(generate_letterboxd_cooccurrence_plot, label="Emotion Co-occurrence Heatmap")

        # r/truefilm Tab
        with gr.TabItem(" r/truefilm", id=2):
            gr.Markdown("### Top Movie Mentions in r/truefilm")
            gr.Plot(generate_truefilm_mentions_plot, label="Top Movie Mentions")

            gr.Markdown("---")
            gr.Markdown("### Average Sentiment Scores per Day in r/truefilm")
            gr.Plot(generate_truefilm_sentiment_plot, label="Daily Sentiment Scores")

            gr.Markdown("---")
            gr.Markdown("### Daily Emotion Trends for a Specific Movie in r/truefilm")
            with gr.Row():
                movie_input_truefilm = gr.Textbox(
                    label="Enter a Movie Name",
                    placeholder="e.g., '2001', 'vertigo', 'persona'...",
                    info="Search for emotion trends of a specific movie"
                )
            movie_sentiment_plot_truefilm = gr.Plot(label="Emotion Trends for Specific Movie")
            movie_input_truefilm.submit(
                generate_truefilm_specific_sentiment_plot,
                inputs=[movie_input_truefilm],
                outputs=[movie_sentiment_plot_truefilm]
            )

            gr.Markdown("---")
            gr.Markdown("### Emotion Co-occurrence Heatmap - r/truefilm")
            gr.Plot(generate_truefilm_cooccurrence_plot, label="Emotion Co-occurrence Heatmap")

        # Comparison Tab
        with gr.TabItem(" Cross-Subreddit Analysis", id=3):
            gr.Markdown("### Emotion Count Comparison Across Subreddits")
            gr.Plot(generate_all_subreddits_emotion_plot, label="Emotion Counts Comparison")

            gr.Markdown("---")
            gr.Markdown("### Average Emotion Scores Across Subreddits")
            gr.Plot(generate_all_subreddits_average_emotion_plot, label="Average Emotion Scores Comparison")
            gr.Markdown(
                """
                *This visualization shows the average sentiment intensity for different emotions
                across all three subreddits, revealing community-specific emotional patterns.*
                """
            )

            gr.Markdown("---")
            gr.Markdown("### Most Popular Movies Across All Subreddits")
            gr.Plot(generate_movie_popularity_comparison, label="Movie Popularity Comparison")
            gr.Markdown(
                """
                *Compare which movies are most discussed across different film communities,
                revealing preferences and trends in each subreddit.*
                """
            )

            gr.Markdown("---")
            gr.Markdown("### Sentiment Score Distribution by Subreddit")
            gr.Plot(generate_sentiment_distribution_plot, label="Sentiment Distribution")
            gr.Markdown(
                """
                *This violin plot shows the distribution and density of emotion scores across subreddits,
                helping identify which communities tend to have more intense or varied emotional responses.*
                """
            )

    # Footer
    gr.Markdown(
        """
        ---

        ### 📈 About This Dashboard - Library Version

        This dashboard provides comprehensive analysis of movie discussions across three distinct Reddit communities using **realistic sample data**:

        - **Data Source**: Generated sample data with 1000+ posts per subreddit
        - **Movies Included**: Popular films across different genres and eras
        - **Emotion Analysis**: 14 different emotions with realistic intensity scores
        - **Interactive Features**: Search for specific movies and explore their emotional patterns
        - **Comparative Analysis**: Cross-community insights and trends

        """
    )

# Launch the dashboard
print("\n Launching Movie Subreddit Insights Dashboard - Library Version...")
print(" Dashboard Features:")
print("   • Individual subreddit analysis (r/movies, r/letterboxd, r/truefilm)")
print("   • Cross-subreddit comparisons")
print("   • Movie-specific sentiment tracking")
print("   • Interactive emotion analysis")
print("   • Popular movie rankings")
print("   • Built-in sample data generation")
print("\n⚡ Starting Gradio interface...")

demo.launch(
    share=True,
    debug=True,
    show_error=True,
    server_name="0.0.0.0",
    height=800
)

print("\n Dashboard launched successfully!")
print(" Use the public URL to access your dashboard from anywhere")
print(" The interface is mobile-friendly and responsive")
print(" No file uploads needed - sample data generated automatically!")

📊 Generating sample datasets...
✅ Sample data generated successfully!
📈 Each dataset contains 1000 posts


/tmp/ipython-input-2092710115.py:628: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend().set_visible(False)  # Remove redundant legend



🚀 Launching Movie Subreddit Insights Dashboard - Library Version...
📊 Dashboard Features:
   • Individual subreddit analysis (r/movies, r/letterboxd, r/truefilm)
   • Cross-subreddit comparisons
   • Movie-specific sentiment tracking
   • Interactive emotion analysis
   • Popular movie rankings
   • Built-in sample data generation

⚡ Starting Gradio interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f7f6aa2dbadc2aed5e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipython-input-2092710115.py:517: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(16, 10))
/tmp/ipython-input-2092710115.py:644: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig = plt.figure(figsize=(14, 8))
/tmp/ipython-input-2092710115.py:628: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend().set